# Notebook 17 — Final Artifact Generation and Validation
**Project:** Respiratory Disease Classification  
**Purpose:** Consolidate *real existing experiment outputs*, generate project reports, and validate artifacts without silently retraining models or inventing results.


## 1. Imports and project paths
This setup uses the repository root and creates only the output directories needed for consolidated reports. Review the resolved path before continuing.

In [1]:
from pathlib import Path
import json
import os
import sys
import platform
import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# Prefer an explicit environment variable when running outside the repository root.
PROJECT_ROOT = Path(os.environ.get("RESPIRATORY_PROJECT_ROOT", Path.cwd())).expanduser().resolve()

# If this notebook is stored under notebooks/, the parent is usually the project root.
if not (PROJECT_ROOT / "notebooks").exists() and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)
print("Root exists:", PROJECT_ROOT.exists())
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root does not exist: {PROJECT_ROOT}")

# These are target locations for final consolidated outputs.
# They do not replace or delete existing project folders.
OUTPUT_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"
for folder in (OUTPUT_DIR, REPORTS_DIR, FIGURES_DIR, TABLES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()

Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Root exists: True


## 2. Read-only artifact discovery
Inventory existing notebooks, model files, tabular results, images, and reports. This cell does not open or execute notebooks, and does not load model weights.

In [2]:
ARTIFACT_EXTENSIONS = {
    ".ipynb", ".joblib", ".pkl", ".pickle", ".pt", ".pth", ".keras", ".h5",
    ".onnx", ".csv", ".json", ".yaml", ".yml", ".txt", ".md", ".pdf",
    ".png", ".jpg", ".jpeg", ".svg", ".parquet"
}
EXCLUDED_DIR_NAMES = {
    ".git", ".venv", "venv", "__pycache__", ".ipynb_checkpoints",
    "node_modules", ".pytest_cache"
}

inventory_rows = []
for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue
    rel_parts = path.relative_to(PROJECT_ROOT).parts
    if any(part in EXCLUDED_DIR_NAMES for part in rel_parts):
        continue
    if path.suffix.lower() not in ARTIFACT_EXTENSIONS:
        continue
    try:
        stat = path.stat()
        inventory_rows.append({
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "extension": path.suffix.lower(),
            "size_bytes": stat.st_size,
            "modified_utc": datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat(),
        })
    except OSError as exc:
        inventory_rows.append({
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "extension": path.suffix.lower(),
            "size_bytes": None,
            "modified_utc": None,
            "inventory_error": str(exc),
        })

inventory_df = pd.DataFrame(inventory_rows)
if not inventory_df.empty:
    inventory_df = inventory_df.sort_values("relative_path").reset_index(drop=True)
display(inventory_df.head(100))
print(f"Discovered {len(inventory_df)} files with recognized artifact extensions.")

,relative_path,extension,size_bytes,modified_utc
0,.vscode/settings.json,.json,194,2026-10-08T10:23:18.647855+00:00
1,README.md,.md,5323,2026-10-09T06:02:22.760436+00:00
2,data/interim/audio_info.csv,.csv,41622,2026-10-08T11:24:28.327778+00:00
3,data/interim/cycles.csv,.csv,983525,2026-10-08T11:40:59.480234+00:00
4,data/interim/cycles_validation.csv,.csv,372830,2026-10-08T11:21:48.460938+00:00
...,...,...,...,...
95,data/processed/deep_learning/xai/gradcam_urti.png,.png,70601,2026-10-08T21:03:16.445521+00:00
96,data/processed/deep_learning/xai/multi_class_x...,.csv,1074,2026-10-08T21:03:24.706665+00:00
97,data/processed/eda/disease_distribution.csv,.csv,104,2026-10-08T14:00:38.714765+00:00
98,data/processed/eda/disease_sound_counts.csv,.csv,200,2026-10-08T14:00:38.716209+00:00


Discovered 1063 files with recognized artifact extensions.


## 3. Categorize discovered files
This is a classification of file paths, not proof that a model is trained or an experiment is valid. Inspect the results before using them in a report.

In [3]:
def categorize_artifact(relative_path):
    p = Path(relative_path)
    low = relative_path.lower()
    suffix = p.suffix.lower()
    if suffix == ".ipynb":
        return "notebook"
    if any(token in low for token in ("prediction", "predictions", "y_pred", "probabilit")):
        return "predictions"
    if any(token in low for token in ("metric", "classification_report", "confusion_matrix", "evaluation")):
        return "evaluation"
    if any(token in low for token in ("history", "loss_curve", "training_log")):
        return "training_history"
    if any(token in low for token in ("model", "checkpoint", "weights", "estimator")) and suffix in {
        ".joblib", ".pkl", ".pickle", ".pt", ".pth", ".keras", ".h5", ".onnx"
    }:
        return "model_or_checkpoint"
    if suffix in {".png", ".jpg", ".jpeg", ".svg"}:
        return "figure"
    if suffix in {".pdf", ".md"}:
        return "report"
    if suffix in {".csv", ".parquet"}:
        return "tabular_data_or_results"
    return "other"

if not inventory_df.empty:
    inventory_df["category"] = inventory_df["relative_path"].map(categorize_artifact)
    display(inventory_df.groupby("category", dropna=False).agg(
        file_count=("relative_path", "count"),
        total_size_bytes=("size_bytes", "sum")
    ).sort_values("file_count", ascending=False))

,file_count,total_size_bytes
category,,
other,938,1741211
tabular_data_or_results,72,13087245
notebook,17,4766815
predictions,12,848876
figure,10,844150
evaluation,7,2057
training_history,4,5921
report,3,23063


## 4. Register actual experiment result files
Use this table to point to **existing, real result files** produced by experiment notebooks. Do not type metric values into this registry by hand.

The registry starts empty intentionally. Add rows only after confirming the files exist and identify the evaluation unit and split. If your existing result CSVs have different schemas, adapt the loading cell below.

In [11]:

#Comprehensive artifact discovery

from pathlib import Path

SEARCH_ROOT = PROJECT_ROOT

EXCLUDED_DIRS = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    ".ipynb_checkpoints",
    "node_modules"
}

MODEL_EXTENSIONS = {
    ".pt", ".pth", ".joblib", ".pkl", ".pickle",
    ".keras", ".h5", ".onnx", ".safetensors"
}

discovered_models = []
discovered_metrics = []
discovered_predictions = []
discovered_histories = []

for path in SEARCH_ROOT.rglob("*"):

    if not path.is_file():
        continue

    relative = path.relative_to(SEARCH_ROOT)

    if any(part in EXCLUDED_DIRS for part in relative.parts):
        continue

    name = path.name.lower()
    suffix = path.suffix.lower()
    relative_path = str(relative)

    if suffix in MODEL_EXTENSIONS:
        discovered_models.append(relative_path)

    if suffix in {".csv", ".json"}:

        if any(term in name for term in [
            "metric",
            "classification_report",
            "confusion_matrix"
        ]):
            discovered_metrics.append(relative_path)

        if any(term in name for term in [
            "prediction",
            "probabilit"
        ]):
            discovered_predictions.append(relative_path)

        if any(term in name for term in [
            "history",
            "training_log"
        ]):
            discovered_histories.append(relative_path)

print("MODEL FILES:", len(discovered_models))
for path in sorted(discovered_models):
    print(" ", path)

print("\\nMETRIC FILES:", len(discovered_metrics))
for path in sorted(discovered_metrics):
    print(" ", path)

print("\\nPREDICTION FILES:", len(discovered_predictions))
for path in sorted(discovered_predictions):
    print(" ", path)

print("\\nTRAINING HISTORY FILES:", len(discovered_histories))
for path in sorted(discovered_histories):
    print(" ", path)


MODEL FILES: 4
  data/processed/deep_learning/cnn_baseline/best_lightweight_mel_cnn.pt
  data/processed/deep_learning/cnn_lstm/best_cnn_lstm.pt
  data/processed/deep_learning/multifeature_cnn/best_multifeature_cnn.pt
  data/processed/deep_learning/multifeature_cnn_12b/best_regularized_multifeature_cnn.pt
\nMETRIC FILES: 9
  data/processed/deep_learning/cnn_baseline/cnn_baseline_metrics.csv
  data/processed/deep_learning/cnn_lstm/cnn_lstm_confusion_matrix.csv
  data/processed/deep_learning/cnn_lstm/cnn_lstm_metrics.csv
  data/processed/deep_learning/multifeature_cnn/multifeature_cnn_confusion_matrix.csv
  data/processed/deep_learning/multifeature_cnn/multifeature_cnn_metrics.csv
  data/processed/deep_learning/multifeature_cnn_12b/confusion_matrix.csv
  data/processed/deep_learning/multifeature_cnn_12b/metrics.csv
  data/processed/deep_learning/patient_level_prediction/majority_vote_confusion_matrix.csv
  data/processed/deep_learning/patient_level_prediction/mean_probability_confusion_ma

In [12]:

from pathlib import Path
import pandas as pd

# Use the PROJECT_ROOT already defined in Notebook 17
base = PROJECT_ROOT / "data" / "processed"

files_to_inspect = [
    "deep_learning/cnn_baseline/cnn_baseline_metrics.csv",
    "deep_learning/cnn_lstm/cnn_lstm_metrics.csv",
    "deep_learning/multifeature_cnn/multifeature_cnn_metrics.csv",
    "deep_learning/multifeature_cnn_12b/metrics.csv",
    "classical_ml/classical_ml_predictions.csv",
    "deep_learning/patient_level_prediction/patient_aggregation_comparison.csv",
]

for relative_path in files_to_inspect:
    path = base / relative_path

    print("\n" + "=" * 90)
    print(f"FILE: {relative_path}")

    if not path.exists():
        print("NOT FOUND — will locate the relevant file separately.")
        continue

    try:
        df = pd.read_csv(path)
        print(f"Shape: {df.shape}")
        print("Columns:", df.columns.tolist())
        print(df.head(8).to_string(index=False))
    except Exception as exc:
        print(f"Could not read file: {exc}")

print("\n" + "=" * 90)
print("OTHER CLASSICAL ML CSV FILES")

classical_dir = base / "classical_ml"
if classical_dir.exists():
    for path in sorted(classical_dir.glob("*.csv")):
        print(f"{path.relative_to(PROJECT_ROOT)}")



FILE: deep_learning/cnn_baseline/cnn_baseline_metrics.csv
Shape: (1, 11)
Columns: ['model', 'accuracy', 'balanced_accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1', 'best_epoch', 'best_validation_loss', 'test_samples', 'test_patients']
              model  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1  weighted_f1  best_epoch  best_validation_loss  test_samples  test_patients
Lightweight Mel CNN  0.716848           0.310168         0.213023      0.265858   0.21288     0.768721          32              1.207017          1561             25

FILE: deep_learning/cnn_lstm/cnn_lstm_metrics.csv
Shape: (1, 12)
Columns: ['model', 'accuracy', 'balanced_accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1', 'test_loss', 'best_epoch', 'best_validation_loss', 'test_samples', 'test_patients']
   model  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1  weighted_f1  test_loss  best_epoch  best_validation_loss  test_sa

In [13]:

import pandas as pd

classical_dir = PROJECT_ROOT / "data" / "processed" / "classical_ml"

files_to_check = [
    "final_classical_ml_comparison.csv",
    "classical_ml_model_comparison.csv",
    "classical_ml_per_disease_results.csv",
]

for filename in files_to_check:
    path = classical_dir / filename

    print("\n" + "=" * 90)
    print(f"FILE: {filename}")

    if not path.exists():
        print("Not found")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(15).to_string(index=False))



FILE: final_classical_ml_comparison.csv
Shape: (3, 8)
Columns: ['Model', 'Accuracy', 'Balanced Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 'Weighted F1', 'ROC AUC (Macro)']
              Model  Accuracy  Balanced Accuracy  Macro Precision  Macro Recall  Macro F1  Weighted F1  ROC AUC (Macro)
      Random Forest  0.874456           0.426495         0.370614      0.266559  0.279604     0.877536              NaN
                SVM  0.806604           0.458256         0.260503      0.286410  0.265580     0.847078              NaN
Logistic Regression  0.641509           0.411920         0.233807      0.257450  0.218218     0.745795              NaN

FILE: classical_ml_model_comparison.csv
Shape: (3, 8)
Columns: ['model', 'accuracy', 'balanced_accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1', 'roc_auc_ovr_macro']
              model  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1  weighted_f1  roc_auc_ovr_macro
      Random Forest 

## 5. Validate registry paths and prevent unsupported claims
The checks below verify paths and flag common reporting issues. They do not prove that a metric was computed correctly; the upstream experiment and evaluation protocol must be reviewed.

In [6]:
def resolve_optional_path(value):
    if pd.isna(value) or not str(value).strip():
        return None
    candidate = Path(str(value)).expanduser()
    return candidate if candidate.is_absolute() else (PROJECT_ROOT / candidate)

registry_checks = []
for _, row in experiment_registry.iterrows():
    item = {"experiment_id": row.get("experiment_id", None)}
    for col in ("metrics_path", "predictions_path", "model_path"):
        p = resolve_optional_path(row.get(col, None))
        item[f"{col}_provided"] = p is not None
        item[f"{col}_exists"] = bool(p is not None and p.is_file())
        if p is not None and not p.is_file():
            item.setdefault("issues", []).append(f"{col} does not exist: {p}")
    unit = str(row.get("evaluation_unit", "")).strip().lower()
    if unit not in {"cycle", "recording", "patient"}:
        item.setdefault("issues", []).append("evaluation_unit must be cycle, recording, or patient")
    if str(row.get("split", "")).strip().lower() != "test":
        item.setdefault("issues", []).append("Not marked as test split; do not present as final test performance")
    if str(row.get("status", "")).strip().lower() != "evaluated":
        item.setdefault("issues", []).append("Experiment status is not evaluated")
    item["issues"] = "; ".join(item.get("issues", []))
    registry_checks.append(item)

registry_check_df = pd.DataFrame(registry_checks)
display(registry_check_df)

""


## 6. Load registered metric files
This loader supports JSON objects and CSV tables. It preserves source file names and experiment identifiers. If a file cannot be parsed, it is reported as an error instead of being silently skipped.

In [7]:
metric_records = []
metric_load_errors = []

for _, row in experiment_registry.iterrows():
    metric_path = resolve_optional_path(row.get("metrics_path", None))
    if metric_path is None:
        continue
    if not metric_path.is_file():
        metric_load_errors.append({"experiment_id": row.get("experiment_id"), "path": str(metric_path), "error": "File not found"})
        continue
    try:
        if metric_path.suffix.lower() == ".csv":
            loaded = pd.read_csv(metric_path)
            loaded["source_file"] = str(metric_path.relative_to(PROJECT_ROOT)) if metric_path.is_relative_to(PROJECT_ROOT) else str(metric_path)
            loaded["experiment_id"] = row.get("experiment_id")
            loaded["task"] = row.get("task")
            loaded["evaluation_unit"] = row.get("evaluation_unit")
            loaded["split"] = row.get("split")
            metric_records.append(loaded)
        elif metric_path.suffix.lower() == ".json":
            with open(metric_path, "r", encoding="utf-8") as f:
                loaded_json = json.load(f)
            metric_records.append(pd.json_normalize(loaded_json).assign(
                source_file=str(metric_path),
                experiment_id=row.get("experiment_id"),
                task=row.get("task"),
                evaluation_unit=row.get("evaluation_unit"),
                split=row.get("split"),
            ))
        else:
            metric_load_errors.append({"experiment_id": row.get("experiment_id"), "path": str(metric_path), "error": "Expected CSV or JSON metrics"})
    except Exception as exc:
        metric_load_errors.append({"experiment_id": row.get("experiment_id"), "path": str(metric_path), "error": repr(exc)})

combined_metrics_df = pd.concat(metric_records, ignore_index=True, sort=False) if metric_records else pd.DataFrame()
display(combined_metrics_df)
display(pd.DataFrame(metric_load_errors))

""


""


## 7. Save the inventory and consolidated metrics
Only discovered files and registered real results are saved. An empty metric table is valid when no results have been registered yet; it is not evidence that experiments achieved zero performance.

In [8]:
inventory_path = REPORTS_DIR / "artifact_inventory.csv"
inventory_df.to_csv(inventory_path, index=False)

if not combined_metrics_df.empty:
    combined_metrics_path = TABLES_DIR / "consolidated_metrics.csv"
    combined_metrics_df.to_csv(combined_metrics_path, index=False)
else:
    combined_metrics_path = None

if not experiment_registry.empty:
    registry_path = REPORTS_DIR / "experiment_registry.csv"
    experiment_registry.to_csv(registry_path, index=False)
else:
    registry_path = None

print("Saved inventory:", inventory_path)
print("Saved consolidated metrics:", combined_metrics_path if combined_metrics_path else "Not generated: no registered metric files")
print("Saved experiment registry:", registry_path if registry_path else "Not generated: registry is empty")

Saved inventory: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/artifact_inventory.csv
Saved consolidated metrics: Not generated: no registered metric files
Saved experiment registry: Not generated: registry is empty


## 8. Generate a project artifact manifest
The manifest records what was found at this run. It does not claim that all experiments have completed.

In [9]:
manifest = {
    "project": "Respiratory Disease Classification",
    "generated_at_utc": RUN_TIMESTAMP,
    "project_root": str(PROJECT_ROOT),
    "python_version": sys.version,
    "platform": platform.platform(),
    "inventory_file": str(inventory_path.relative_to(PROJECT_ROOT)),
    "inventory_file_count": int(len(inventory_df)),
    "registered_experiment_count": int(len(experiment_registry)),
    "registered_metric_rows_loaded": int(len(combined_metrics_df)),
    "metric_load_error_count": int(len(metric_load_errors)),
    "notes": [
        "No model training was performed by this notebook.",
        "File discovery does not verify scientific validity of an experiment.",
        "Cycle-level, recording-level, and patient-level evaluations must remain distinct.",
        "Only registered and successfully loaded metric files are included in consolidated metrics."
    ]
}
manifest_path = REPORTS_DIR / "experiment_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print("Saved manifest:", manifest_path)
print(json.dumps(manifest, indent=2))

Saved manifest: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/experiment_manifest.json
{
  "project": "Respiratory Disease Classification",
  "generated_at_utc": "2026-10-09T18:04:52.384874+00:00",
  "project_root": "/Users/vs/Sem 5/Respiratory-Disease-Classification",
  "python_version": "3.14.7 (main, Aug  5 2026, 10:29:49) [Clang 21.0.0 (clang-2100.1.1.101)]",
  "platform": "macOS-27.0.1-arm64-arm-64bit-Mach-O",
  "inventory_file": "reports/artifact_inventory.csv",
  "inventory_file_count": 1063,
  "registered_experiment_count": 0,
  "registered_metric_rows_loaded": 0,
  "metric_load_error_count": 0,
  "notes": [
    "No model training was performed by this notebook.",
    "File discovery does not verify scientific validity of an experiment.",
    "Cycle-level, recording-level, and patient-level evaluations must remain distinct.",
    "Only registered and successfully loaded metric files are included in consolidated metrics."
  ]
}


## 9. Create a final validation summary
This summary is deliberately conservative: empty registries and missing outputs are surfaced as action items, not treated as success.

In [10]:
summary = {
    "project_root_exists": PROJECT_ROOT.exists(),
    "inventory_created": inventory_path.is_file(),
    "manifest_created": manifest_path.is_file(),
    "discovered_model_or_checkpoint_files": int(
        (inventory_df["category"] == "model_or_checkpoint").sum()
    ) if not inventory_df.empty and "category" in inventory_df.columns else 0,
    "discovered_prediction_files": int(
        (inventory_df["category"] == "predictions").sum()
    ) if not inventory_df.empty and "category" in inventory_df.columns else 0,
    "registered_experiments": int(len(experiment_registry)),
    "loaded_metric_rows": int(len(combined_metrics_df)),
    "metric_load_errors": int(len(metric_load_errors)),
}

if summary["registered_experiments"] == 0:
    summary["status"] = "INCOMPLETE: register existing experiment outputs"
elif summary["metric_load_errors"] > 0:
    summary["status"] = "NEEDS ATTENTION: fix metric loading errors"
else:
    summary["status"] = "DISCOVERY COMPLETE: manually review scientific and split validity"

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))

,value
project_root_exists,True
inventory_created,True
manifest_created,True
discovered_model_or_checkpoint_files,0
discovered_prediction_files,12
registered_experiments,0
loaded_metric_rows,0
metric_load_errors,0
status,INCOMPLETE: register existing experiment outputs


## 10. Next actions
1. Review the inventory and map real experiment outputs to `experiment_registry`.
2. For any experiment with no saved model/predictions/metrics, return to its existing experiment notebook and add saving code there.
3. Keep final test predictions separate from training and validation predictions.
4. Run the repository test suite separately with `pytest`.
5. Review all reported metrics and the split protocol before using them in the final academic report.

**Do not report final model performance until the relevant evaluation has actually run and the test split protocol has been confirmed.**

In [14]:

from pathlib import Path
import pandas as pd
import json

base = PROJECT_ROOT / "data" / "processed"
output_dir = PROJECT_ROOT / "reports" / "final_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 1. Consolidate cycle-level model metrics from saved files
# ---------------------------------------------------------
metric_sources = [
    ("Classical ML", "classical_ml/final_classical_ml_comparison.csv"),
    ("Deep Learning", "deep_learning/cnn_baseline/cnn_baseline_metrics.csv"),
    ("Deep Learning", "deep_learning/cnn_lstm/cnn_lstm_metrics.csv"),
    ("Deep Learning", "deep_learning/multifeature_cnn/multifeature_cnn_metrics.csv"),
    ("Deep Learning", "deep_learning/multifeature_cnn_12b/metrics.csv"),
]

frames = []

for model_family, relative_path in metric_sources:
    path = base / relative_path

    if not path.exists():
        print(f"MISSING: {relative_path}")
        continue

    df = pd.read_csv(path)

    # Normalize column names without changing the source files.
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]

    if "model" not in df.columns:
        print(f"SKIPPED — no model column: {relative_path}")
        continue

    df["model_family"] = model_family
    df["source_file"] = relative_path
    df["evaluation_unit"] = "cycle"
    df["split_comparability"] = "UNVERIFIED — confirm identical test patients/cycles"

    frames.append(df)

if frames:
    comparison = pd.concat(frames, ignore_index=True, sort=False)

    # Put the most useful fields first; retain any additional source columns.
    preferred = [
        "model_family", "model", "evaluation_unit",
        "accuracy", "balanced_accuracy", "macro_precision",
        "macro_recall", "macro_f1", "weighted_f1",
        "roc_auc_(macro)", "roc_auc_ovr_macro",
        "test_samples", "test_patients", "best_epoch",
        "best_validation_loss", "test_loss",
        "split_comparability", "source_file",
    ]
    ordered = [c for c in preferred if c in comparison.columns]
    ordered += [c for c in comparison.columns if c not in ordered]
    comparison = comparison[ordered]

    comparison_path = output_dir / "cycle_level_model_comparison.csv"
    comparison.to_csv(comparison_path, index=False)

    print("Saved:", comparison_path)
    print("Rows:", len(comparison))
    display(comparison)
else:
    print("No metric files were loaded. Check PROJECT_ROOT and source paths.")

# ---------------------------------------------------------
# 2. Save patient-level evaluation separately
# ---------------------------------------------------------
patient_path = (
    base / "deep_learning" / "patient_level_prediction"
    / "patient_aggregation_comparison.csv"
)

if patient_path.exists():
    patient_results = pd.read_csv(patient_path)
    patient_results["evaluation_unit"] = "patient"
    patient_results["source_file"] = str(
        patient_path.relative_to(PROJECT_ROOT)
    )

    patient_output = output_dir / "patient_level_model_comparison.csv"
    patient_results.to_csv(patient_output, index=False)

    print("\nSaved:", patient_output)
    display(patient_results)
else:
    print("Patient-level aggregation comparison was not found.")

# ---------------------------------------------------------
# 3. Write a concise manifest of generated outputs
# ---------------------------------------------------------
manifest = {
    "project": "Respiratory Disease Classification",
    "generated_from_existing_results": True,
    "retraining_performed": False,
    "outputs": [],
    "important_caveat": (
        "Cycle-level results are consolidated for review, but direct "
        "model ranking requires verification that the models used the "
        "same held-out test patients and cycles. Patient-level results "
        "are stored separately."
    ),
}

for filename in [
    "cycle_level_model_comparison.csv",
    "patient_level_model_comparison.csv",
]:
    path = output_dir / filename
    if path.exists():
        manifest["outputs"].append({
            "file": filename,
            "rows": len(pd.read_csv(path)),
            "size_bytes": path.stat().st_size,
        })

manifest_path = output_dir / "comparison_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\nSaved:", manifest_path)
print("Consolidation complete.")


Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/cycle_level_model_comparison.csv
Rows: 7


,model_family,model,evaluation_unit,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,roc_auc_(macro),test_samples,test_patients,best_epoch,best_validation_loss,test_loss,split_comparability,source_file
0,Classical ML,Random Forest,cycle,0.874456,0.426495,0.370614,0.266559,0.279604,0.877536,NaN,NaN,NaN,NaN,NaN,NaN,UNVERIFIED — confirm identical test patients/c...,classical_ml/final_classical_ml_comparison.csv
1,Classical ML,SVM,cycle,0.806604,0.458256,0.260503,0.286410,0.265580,0.847078,NaN,NaN,NaN,NaN,NaN,NaN,UNVERIFIED — confirm identical test patients/c...,classical_ml/final_classical_ml_comparison.csv
2,Classical ML,Logistic Regression,cycle,0.641509,0.411920,0.233807,0.257450,0.218218,0.745795,NaN,NaN,NaN,NaN,NaN,NaN,UNVERIFIED — confirm identical test patients/c...,classical_ml/final_classical_ml_comparison.csv
3,Deep Learning,Lightweight Mel CNN,cycle,0.716848,0.310168,0.213023,0.265858,0.212880,0.768721,NaN,1561.0,25.0,32.0,1.207017,NaN,UNVERIFIED — confirm identical test patients/c...,deep_learning/cnn_baseline/cnn_baseline_metric...
4,Deep Learning,CNN-LSTM,cycle,0.813581,0.226085,0.134473,0.169564,0.146471,0.786735,NaN,1561.0,25.0,8.0,1.271949,0.861682,UNVERIFIED — confirm identical test patients/c...,deep_learning/cnn_lstm/cnn_lstm_metrics.csv
5,Deep Learning,Multi-Feature CNN,cycle,0.855221,0.166417,0.142720,0.166417,0.153660,0.789662,NaN,1561.0,NaN,1.0,2.452791,NaN,UNVERIFIED — confirm identical test patients/c...,deep_learning/multifeature_cnn/multifeature_cn...
6,Deep Learning,Regularized Multi-Feature CNN (12B),cycle,0.856502,0.166667,0.142750,0.166667,0.153784,0.790299,NaN,1561.0,NaN,2.0,1.332201,NaN,UNVERIFIED — confirm identical test patients/c...,deep_learning/multifeature_cnn_12b/metrics.csv



Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/patient_level_model_comparison.csv


,rank,aggregation_method,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,evaluation_unit,source_file
0,1,Majority Vote,0.56,0.333333,0.132812,0.25,0.162069,0.482207,patient,data/processed/deep_learning/patient_level_pre...
1,2,Mean Probability,0.56,0.333333,0.132812,0.25,0.162069,0.482207,patient,data/processed/deep_learning/patient_level_pre...



Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/comparison_manifest.json
Consolidation complete.


In [15]:

from pathlib import Path
import pandas as pd
import json

base = PROJECT_ROOT / "data" / "processed" / "deep_learning"
output_dir = PROJECT_ROOT / "reports" / "final_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

experiments = [
    {
        "experiment": "Lightweight Mel CNN",
        "folder": "cnn_baseline",
        "checkpoint": "best_lightweight_mel_cnn.pt",
        "metrics": "cnn_baseline_metrics.csv",
        "predictions": "cnn_test_predictions.csv",
        "history": "cnn_training_history.csv",
    },
    {
        "experiment": "CNN-LSTM",
        "folder": "cnn_lstm",
        "checkpoint": "best_cnn_lstm.pt",
        "metrics": "cnn_lstm_metrics.csv",
        "predictions": "cnn_lstm_test_predictions.csv",
        "history": "cnn_lstm_training_history.csv",
    },
    {
        "experiment": "Multi-Feature CNN",
        "folder": "multifeature_cnn",
        "checkpoint": "best_multifeature_cnn.pt",
        "metrics": "multifeature_cnn_metrics.csv",
        "predictions": "multifeature_cnn_test_predictions.csv",
        "history": "multifeature_cnn_training_history.csv",
    },
    {
        "experiment": "Regularized Multi-Feature CNN (12B)",
        "folder": "multifeature_cnn_12b",
        "checkpoint": "best_regularized_multifeature_cnn.pt",
        "metrics": "metrics.csv",
        "predictions": "test_predictions.csv",
        "history": "training_history.csv",
    },
]

registry_rows = []

for exp in experiments:
    folder = base / exp["folder"]

    row = {
        "experiment": exp["experiment"],
        "checkpoint_path": str(folder / exp["checkpoint"]),
        "metrics_path": str(folder / exp["metrics"]),
        "predictions_path": str(folder / exp["predictions"]),
        "training_history_path": str(folder / exp["history"]),
    }

    for key in ["checkpoint", "metrics", "predictions", "history"]:
        row[f"{key}_exists"] = (folder / exp[key]).is_file()

    row["all_artifacts_present"] = all(
        row[f"{key}_exists"]
        for key in ["checkpoint", "metrics", "predictions", "history"]
    )

    registry_rows.append(row)

registry = pd.DataFrame(registry_rows)

registry_path = output_dir / "model_artifact_registry.csv"
registry.to_csv(registry_path, index=False)

print("Saved registry:", registry_path)
display(registry)

print("\nMissing artifacts:")
missing = registry[~registry["all_artifacts_present"]]
if missing.empty:
    print("None — all four registered artifact types exist for all four experiments.")
else:
    display(missing[[
        "experiment",
        "checkpoint_exists",
        "metrics_exists",
        "predictions_exists",
        "history_exists",
    ]])

# Save a JSON manifest with file sizes and existence checks.
manifest = []

for _, row in registry.iterrows():
    for kind in ["checkpoint", "metrics", "predictions", "training_history"]:
        path_key = f"{kind}_path"
        path = Path(row[path_key])

        manifest.append({
            "experiment": row["experiment"],
            "artifact_type": kind,
            "path": str(path),
            "exists": path.is_file(),
            "size_bytes": path.stat().st_size if path.is_file() else None,
        })

manifest_path = output_dir / "artifact_file_manifest.csv"
pd.DataFrame(manifest).to_csv(manifest_path, index=False)

print("\nSaved file manifest:", manifest_path)


Saved registry: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/model_artifact_registry.csv


,experiment,checkpoint_path,metrics_path,predictions_path,training_history_path,checkpoint_exists,metrics_exists,predictions_exists,history_exists,all_artifacts_present
0,Lightweight Mel CNN,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,True,True,True,True,True
1,CNN-LSTM,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,True,True,True,True,True
2,Multi-Feature CNN,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,True,True,True,True,True
3,Regularized Multi-Feature CNN (12B),/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,True,True,True,True,True



Missing artifacts:
None — all four registered artifact types exist for all four experiments.

Saved file manifest: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/artifact_file_manifest.csv


In [16]:

from pathlib import Path
import pandas as pd
import json

processed_dir = PROJECT_ROOT / "data" / "processed"
output_dir = PROJECT_ROOT / "reports" / "final_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

prediction_files = [
    ("Lightweight Mel CNN",
     "deep_learning/cnn_baseline/cnn_test_predictions.csv"),
    ("CNN-LSTM",
     "deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv"),
    ("Multi-Feature CNN",
     "deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv"),
    ("Regularized Multi-Feature CNN (12B)",
     "deep_learning/multifeature_cnn_12b/test_predictions.csv"),
    ("Classical ML",
     "classical_ml/classical_ml_predictions.csv"),
    ("Patient Majority Vote",
     "deep_learning/patient_level_prediction/patient_majority_predictions.csv"),
    ("Patient Probability",
     "deep_learning/patient_level_prediction/patient_probability_predictions.csv"),
]

validation_rows = []

for model_name, relative_path in prediction_files:
    path = processed_dir / relative_path

    result = {
        "model": model_name,
        "file": relative_path,
        "exists": path.is_file(),
        "rows": None,
        "columns": None,
        "missing_cells": None,
        "has_true_label_column": False,
        "has_prediction_column": False,
        "status": "",
    }

    if not path.is_file():
        result["status"] = "MISSING FILE"
        validation_rows.append(result)
        continue

    try:
        df = pd.read_csv(path)
        columns = df.columns.tolist()

        result["rows"] = len(df)
        result["columns"] = ", ".join(map(str, columns))
        result["missing_cells"] = int(df.isna().sum().sum())

        lower_to_original = {str(c).lower(): c for c in columns}

        true_candidates = [
            "true_diagnosis", "true_label", "y_true",
            "actual", "actual_label", "target", "label"
        ]
        pred_candidates = [
            "prediction", "predicted_label", "y_pred",
            "predicted", "majority_prediction",
            "mean_probability_prediction"
        ]

        result["has_true_label_column"] = any(
            c in lower_to_original for c in true_candidates
        )
        result["has_prediction_column"] = any(
            c in lower_to_original for c in pred_candidates
        )

        if df.empty:
            result["status"] = "FAIL: EMPTY FILE"
        elif result["missing_cells"] > 0:
            result["status"] = "REVIEW: MISSING CELLS"
        else:
            result["status"] = "PASS: BASIC FILE CHECKS"

    except Exception as exc:
        result["status"] = f"FAIL: {type(exc).__name__}: {exc}"

    validation_rows.append(result)

validation_df = pd.DataFrame(validation_rows)
validation_path = output_dir / "prediction_validation.csv"
validation_df.to_csv(validation_path, index=False)

print("Saved:", validation_path)
display(validation_df[[
    "model", "exists", "rows", "missing_cells",
    "has_true_label_column", "has_prediction_column", "status"
]])

# ---------------------------------------------------------
# Compare deep-learning prediction row counts with metrics
# ---------------------------------------------------------
metric_files = [
    ("Lightweight Mel CNN",
     "deep_learning/cnn_baseline/cnn_baseline_metrics.csv",
     "deep_learning/cnn_baseline/cnn_test_predictions.csv"),
    ("CNN-LSTM",
     "deep_learning/cnn_lstm/cnn_lstm_metrics.csv",
     "deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv"),
    ("Multi-Feature CNN",
     "deep_learning/multifeature_cnn/multifeature_cnn_metrics.csv",
     "deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv"),
    ("Regularized Multi-Feature CNN (12B)",
     "deep_learning/multifeature_cnn_12b/metrics.csv",
     "deep_learning/multifeature_cnn_12b/test_predictions.csv"),
]

count_checks = []

for model_name, metric_rel, prediction_rel in metric_files:
    metric_path = processed_dir / metric_rel
    prediction_path = processed_dir / prediction_rel

    row = {
        "model": model_name,
        "metric_file_exists": metric_path.is_file(),
        "prediction_file_exists": prediction_path.is_file(),
        "reported_test_samples": None,
        "prediction_rows": None,
        "counts_match": None,
        "status": "",
    }

    if not metric_path.is_file() or not prediction_path.is_file():
        row["status"] = "REVIEW: FILE MISSING"
        count_checks.append(row)
        continue

    metrics = pd.read_csv(metric_path)
    predictions = pd.read_csv(prediction_path)

    if "test_samples" in metrics.columns and len(metrics) > 0:
        row["reported_test_samples"] = int(metrics.iloc[0]["test_samples"])
        row["prediction_rows"] = len(predictions)
        row["counts_match"] = (
            row["reported_test_samples"] == row["prediction_rows"]
        )
        row["status"] = "PASS" if row["counts_match"] else "REVIEW: COUNT MISMATCH"
    else:
        row["prediction_rows"] = len(predictions)
        row["status"] = "REVIEW: NO test_samples FIELD"

    count_checks.append(row)

count_df = pd.DataFrame(count_checks)
count_path = output_dir / "prediction_metric_count_checks.csv"
count_df.to_csv(count_path, index=False)

print("\nSaved:", count_path)
display(count_df)

print("\nValidation files saved under:", output_dir)


Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/prediction_validation.csv


,model,exists,rows,missing_cells,has_true_label_column,has_prediction_column,status
0,Lightweight Mel CNN,True,1561,0,True,True,PASS: BASIC FILE CHECKS
1,CNN-LSTM,True,1561,0,False,False,PASS: BASIC FILE CHECKS
2,Multi-Feature CNN,True,1561,0,True,True,PASS: BASIC FILE CHECKS
3,Regularized Multi-Feature CNN (12B),True,1561,0,True,True,PASS: BASIC FILE CHECKS
4,Classical ML,True,2756,0,True,False,PASS: BASIC FILE CHECKS
5,Patient Majority Vote,True,25,0,True,True,PASS: BASIC FILE CHECKS
6,Patient Probability,True,25,0,True,True,PASS: BASIC FILE CHECKS



Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/prediction_metric_count_checks.csv


,model,metric_file_exists,prediction_file_exists,reported_test_samples,prediction_rows,counts_match,status
0,Lightweight Mel CNN,True,True,1561,1561,True,PASS
1,CNN-LSTM,True,True,1561,1561,True,PASS
2,Multi-Feature CNN,True,True,1561,1561,True,PASS
3,Regularized Multi-Feature CNN (12B),True,True,1561,1561,True,PASS



Validation files saved under: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts


In [18]:

from pathlib import Path
import pandas as pd

# Reuse the project root already defined in Notebook 17
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS = PROJECT_ROOT / "reports" / "final_artifacts"

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.is_dir())
print("Processed data exists:", PROCESSED.is_dir())
print("Final artifacts directory exists:", ARTIFACTS.is_dir())


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Project root exists: True
Processed data exists: True
Final artifacts directory exists: True


In [20]:

from pathlib import Path
import pandas as pd

# Explicit project path — no __file__ required in Jupyter
PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
).resolve()

PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS = PROJECT_ROOT / "reports" / "final_artifacts"

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.is_dir())
print("Processed data exists:", PROCESSED.is_dir())
print("Final artifacts directory exists:", ARTIFACTS.is_dir())

assert PROJECT_ROOT.is_dir(), (
    "Project folder not found. Check the project path."
)
assert PROCESSED.is_dir(), (
    "The data/processed folder was not found."
)


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Project root exists: True
Processed data exists: True
Final artifacts directory exists: True


In [21]:

from pathlib import Path
import pandas as pd
import torch

PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
).resolve()

checkpoint_specs = [
    {
        "model": "Lightweight Mel CNN",
        "path": "data/processed/deep_learning/cnn_baseline/best_lightweight_mel_cnn.pt",
    },
    {
        "model": "CNN-LSTM",
        "path": "data/processed/deep_learning/cnn_lstm/best_cnn_lstm.pt",
    },
    {
        "model": "Multi-Feature CNN",
        "path": "data/processed/deep_learning/multifeature_cnn/best_multifeature_cnn.pt",
    },
    {
        "model": "Regularized Multi-Feature CNN (12B)",
        "path": "data/processed/deep_learning/multifeature_cnn_12b/best_regularized_multifeature_cnn.pt",
    },
]

checkpoint_results = []

for spec in checkpoint_specs:
    path = PROJECT_ROOT / spec["path"]

    print("\n" + "=" * 75)
    print("MODEL:", spec["model"])
    print("PATH:", path)

    result = {
        "model": spec["model"],
        "path": spec["path"],
        "exists": path.is_file(),
        "size_bytes": path.stat().st_size if path.is_file() else None,
        "load_status": "NOT CHECKED",
        "checkpoint_type": None,
        "checkpoint_keys": None,
        "tensor_count": None,
        "error": None,
    }

    if not path.is_file():
        result["load_status"] = "MISSING"
        print("ERROR: checkpoint file not found")
        checkpoint_results.append(result)
        continue

    try:
        # Load on CPU and use restricted loading where supported.
        try:
            checkpoint = torch.load(
                path, map_location="cpu", weights_only=True
            )
        except TypeError:
            # Compatibility fallback for older PyTorch versions.
            checkpoint = torch.load(path, map_location="cpu")

        result["checkpoint_type"] = type(checkpoint).__name__

        if isinstance(checkpoint, dict):
            keys = list(checkpoint.keys())
            result["checkpoint_keys"] = ", ".join(map(str, keys))
            print("Checkpoint keys:", keys)

            # Detect common checkpoint layouts.
            state_dict = None
            for key in (
                "model_state_dict",
                "state_dict",
                "model",
                "weights",
            ):
                candidate = checkpoint.get(key)
                if isinstance(candidate, dict):
                    state_dict = candidate
                    print("Model weights found under:", key)
                    break

            # Some files contain the state_dict directly.
            if state_dict is None and checkpoint and all(
                isinstance(value, torch.Tensor)
                for value in checkpoint.values()
            ):
                state_dict = checkpoint
                print("Checkpoint appears to be a raw state_dict.")

            if state_dict is not None:
                tensor_items = [
                    (key, value)
                    for key, value in state_dict.items()
                    if isinstance(value, torch.Tensor)
                ]
                result["tensor_count"] = len(tensor_items)
                print("Tensor count:", len(tensor_items))

                for key, tensor in tensor_items[:12]:
                    print(f"  {key}: shape={tuple(tensor.shape)}")

                if tensor_items:
                    result["load_status"] = "PASS: WEIGHT TENSORS READ"
                else:
                    result["load_status"] = "REVIEW: NO TENSORS FOUND"
            else:
                result["load_status"] = "REVIEW: UNKNOWN CHECKPOINT STRUCTURE"

        else:
            result["load_status"] = "REVIEW: NOT A DICTIONARY CHECKPOINT"
            print("Loaded object type:", type(checkpoint).__name__)

    except Exception as exc:
        result["load_status"] = "FAIL: LOAD ERROR"
        result["error"] = f"{type(exc).__name__}: {exc}"
        print("Checkpoint load error:", result["error"])

    checkpoint_results.append(result)

checkpoint_df = pd.DataFrame(checkpoint_results)

report_dir = PROJECT_ROOT / "reports" / "final_artifacts"
report_dir.mkdir(parents=True, exist_ok=True)

report_path = report_dir / "checkpoint_integrity_check.csv"
checkpoint_df.to_csv(report_path, index=False)

print("\n" + "=" * 75)
print("CHECKPOINT INTEGRITY SUMMARY")
display(checkpoint_df)

print("Saved report:", report_path)



MODEL: Lightweight Mel CNN
PATH: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_baseline/best_lightweight_mel_cnn.pt
Checkpoint keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_loss', 'val_accuracy', 'class_mapping', 'config']
Model weights found under: model_state_dict
Tensor count: 23
  features.0.weight: shape=(16, 1, 3, 3)
  features.0.bias: shape=(16,)
  features.1.weight: shape=(16,)
  features.1.bias: shape=(16,)
  features.1.running_mean: shape=(16,)
  features.1.running_var: shape=(16,)
  features.1.num_batches_tracked: shape=()
  features.4.weight: shape=(32, 16, 3, 3)
  features.4.bias: shape=(32,)
  features.5.weight: shape=(32,)
  features.5.bias: shape=(32,)
  features.5.running_mean: shape=(32,)

MODEL: CNN-LSTM
PATH: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_lstm/best_cnn_lstm.pt
Checkpoint keys: ['cnn.0.weight', 'cnn.0.bias', 'cnn.1.weight', 'cnn.1.bias', 'cnn.1.running_mean

,model,path,exists,size_bytes,load_status,checkpoint_type,checkpoint_keys,tensor_count,error
0,Lightweight Mel CNN,data/processed/deep_learning/cnn_baseline/best...,True,310935,PASS: WEIGHT TENSORS READ,dict,"epoch, model_state_dict, optimizer_state_dict,...",23,None
1,CNN-LSTM,data/processed/deep_learning/cnn_lstm/best_cnn...,True,373412,PASS: WEIGHT TENSORS READ,OrderedDict,"cnn.0.weight, cnn.0.bias, cnn.1.weight, cnn.1....",31,None
2,Multi-Feature CNN,data/processed/deep_learning/multifeature_cnn/...,True,464538,PASS: WEIGHT TENSORS READ,OrderedDict,"logmel_branch.features.0.weight, logmel_branch...",73,None
3,Regularized Multi-Feature CNN (12B),data/processed/deep_learning/multifeature_cnn_...,True,465550,PASS: WEIGHT TENSORS READ,OrderedDict,"logmel_branch.features.0.weight, logmel_branch...",73,None


Saved report: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/checkpoint_integrity_check.csv


In [22]:

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
)

PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
).resolve()
processed = PROJECT_ROOT / "data" / "processed"
output_dir = PROJECT_ROOT / "reports" / "final_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

experiments = [
    ("Lightweight Mel CNN",
     "deep_learning/cnn_baseline/cnn_test_predictions.csv",
     "deep_learning/cnn_baseline/cnn_baseline_metrics.csv"),
    ("CNN-LSTM",
     "deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv",
     "deep_learning/cnn_lstm/cnn_lstm_metrics.csv"),
    ("Multi-Feature CNN",
     "deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv",
     "deep_learning/multifeature_cnn/multifeature_cnn_metrics.csv"),
    ("Regularized Multi-Feature CNN (12B)",
     "deep_learning/multifeature_cnn_12b/test_predictions.csv",
     "deep_learning/multifeature_cnn_12b/metrics.csv"),
]

def locate_column(df, candidates):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    return next((lookup[c] for c in candidates if c in lookup), None)

true_candidates = [
    "true_label", "true_diagnosis", "y_true", "label", "actual",
    "target", "true_class"
]
pred_candidates = [
    "prediction", "predicted_label", "y_pred", "predicted",
    "predicted_class", "predicted_diagnosis"
]

rows = []

for model, pred_rel, metric_rel in experiments:
    pred_path = processed / pred_rel
    metric_path = processed / metric_rel

    row = {
        "model": model,
        "prediction_rows": None,
        "true_column": None,
        "prediction_column": None,
        "recomputed_accuracy": None,
        "saved_accuracy": None,
        "recomputed_balanced_accuracy": None,
        "saved_balanced_accuracy": None,
        "recomputed_macro_f1": None,
        "saved_macro_f1": None,
        "status": "",
        "details": "",
    }

    if not pred_path.is_file() or not metric_path.is_file():
        row["status"] = "REVIEW: missing prediction or metric file"
        rows.append(row)
        continue

    predictions = pd.read_csv(pred_path)
    metrics = pd.read_csv(metric_path)

    true_col = locate_column(predictions, true_candidates)
    pred_col = locate_column(predictions, pred_candidates)

    row["prediction_rows"] = len(predictions)
    row["true_column"] = true_col
    row["prediction_column"] = pred_col

    if true_col is None or pred_col is None:
        row["status"] = "REVIEW: label columns not identified"
        row["details"] = f"Available columns: {list(predictions.columns)}"
        rows.append(row)
        continue

    pairs = predictions[[true_col, pred_col]].dropna()
    if pairs.empty:
        row["status"] = "REVIEW: no valid label pairs"
        rows.append(row)
        continue

    y_true = pairs[true_col].astype(str)
    y_pred = pairs[pred_col].astype(str)

    row["recomputed_accuracy"] = accuracy_score(y_true, y_pred)
    row["recomputed_balanced_accuracy"] = balanced_accuracy_score(
        y_true, y_pred
    )
    row["recomputed_macro_f1"] = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )[2]

    saved = metrics.iloc[0]
    comparisons = []

    for key in ["accuracy", "balanced_accuracy", "macro_f1"]:
        if key not in metrics.columns:
            comparisons.append(f"{key}: absent from saved metrics")
            continue

        calculated = row[f"recomputed_{key}"]
        saved_value = pd.to_numeric(
            pd.Series([saved[key]]), errors="coerce"
        ).iloc[0]

        row[f"saved_{key}"] = saved_value

        if pd.isna(saved_value):
            comparisons.append(f"{key}: saved value is missing")
        elif not np.isclose(calculated, saved_value, atol=1e-5):
            comparisons.append(
                f"{key}: recomputed={calculated:.6f}, saved={saved_value:.6f}"
            )

    row["status"] = (
        "PASS: metrics reproduced"
        if not comparisons
        else "REVIEW: differences or missing fields"
    )
    row["details"] = "; ".join(comparisons)
    rows.append(row)

result = pd.DataFrame(rows)
report_path = output_dir / "prediction_metric_reproduction_check.csv"
result.to_csv(report_path, index=False)

display(result)
print("Saved report:", report_path)


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,model,prediction_rows,true_column,prediction_column,recomputed_accuracy,saved_accuracy,recomputed_balanced_accuracy,saved_balanced_accuracy,recomputed_macro_f1,saved_macro_f1,status,details
0,Lightweight Mel CNN,1561,true_label,predicted_label,0.716848,0.716848,0.310168,0.310168,0.212880,0.212880,PASS: metrics reproduced,
1,CNN-LSTM,1561,NaN,predicted_diagnosis,NaN,NaN,NaN,NaN,NaN,NaN,REVIEW: label columns not identified,"Available columns: ['patient_id', 'recording_i..."
2,Multi-Feature CNN,1561,true_label,predicted_label,0.855221,0.855221,0.166417,0.166417,0.153660,0.153660,PASS: metrics reproduced,
3,Regularized Multi-Feature CNN (12B),1561,true_label,predicted_label,0.856502,0.856502,0.166667,0.166667,0.153784,0.153784,PASS: metrics reproduced,


Saved report: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/prediction_metric_reproduction_check.csv


In [23]:

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
).resolve()

path = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "cnn_lstm" / "cnn_lstm_test_predictions.csv"
)

df = pd.read_csv(path)

print("Columns:")
print(df.columns.tolist())

print("\nFirst 10 rows:")
display(df.head(10))

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())


Columns:
['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split', 'predicted_disease_label', 'predicted_diagnosis']

First 10 rows:


,patient_id,recording_id,chest_location,diagnosis,disease_label,official_split,audio_path,cnn_split,predicted_disease_label,predicted_diagnosis
0,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
1,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
2,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
3,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
4,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
5,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
6,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
7,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
8,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD
9,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test,3,COPD



Data types:
patient_id                 int64
recording_id                 str
chest_location               str
diagnosis                    str
disease_label              int64
official_split               str
audio_path                   str
cnn_split                    str
predicted_disease_label    int64
predicted_diagnosis          str
dtype: object

Missing values:
patient_id                 0
recording_id               0
chest_location             0
diagnosis                  0
disease_label              0
official_split             0
audio_path                 0
cnn_split                  0
predicted_disease_label    0
predicted_diagnosis        0
dtype: int64


In [24]:

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
)

PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
).resolve()
processed = PROJECT_ROOT / "data" / "processed"
output_dir = PROJECT_ROOT / "reports" / "final_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

experiments = [
    {
        "model": "Lightweight Mel CNN",
        "pred": "deep_learning/cnn_baseline/cnn_test_predictions.csv",
        "metrics": "deep_learning/cnn_baseline/cnn_baseline_metrics.csv",
        "true": "true_label",
        "prediction": "predicted_label",
    },
    {
        "model": "CNN-LSTM",
        "pred": "deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv",
        "metrics": "deep_learning/cnn_lstm/cnn_lstm_metrics.csv",
        "true": "diagnosis",
        "prediction": "predicted_diagnosis",
    },
    {
        "model": "Multi-Feature CNN",
        "pred": "deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv",
        "metrics": "deep_learning/multifeature_cnn/multifeature_cnn_metrics.csv",
        "true": "true_label",
        "prediction": "predicted_label",
    },
    {
        "model": "Regularized Multi-Feature CNN (12B)",
        "pred": "deep_learning/multifeature_cnn_12b/test_predictions.csv",
        "metrics": "deep_learning/multifeature_cnn_12b/metrics.csv",
        "true": "true_label",
        "prediction": "predicted_label",
    },
]

results = []

for exp in experiments:
    pred_path = processed / exp["pred"]
    metric_path = processed / exp["metrics"]

    row = {
        "model": exp["model"],
        "prediction_rows": None,
        "recomputed_accuracy": None,
        "saved_accuracy": None,
        "recomputed_balanced_accuracy": None,
        "saved_balanced_accuracy": None,
        "recomputed_macro_f1": None,
        "saved_macro_f1": None,
        "status": "",
        "details": "",
    }

    if not pred_path.is_file() or not metric_path.is_file():
        row["status"] = "REVIEW: FILE MISSING"
        results.append(row)
        continue

    pred_df = pd.read_csv(pred_path)
    metric_df = pd.read_csv(metric_path)

    true_col = exp["true"]
    pred_col = exp["prediction"]

    if true_col not in pred_df.columns or pred_col not in pred_df.columns:
        row["status"] = "REVIEW: EXPECTED COLUMN MISSING"
        row["details"] = f"Available columns: {pred_df.columns.tolist()}"
        results.append(row)
        continue

    pairs = pred_df[[true_col, pred_col]].dropna()
    row["prediction_rows"] = len(pred_df)

    if pairs.empty:
        row["status"] = "REVIEW: NO VALID LABEL PAIRS"
        results.append(row)
        continue

    y_true = pairs[true_col].astype(str)
    y_pred = pairs[pred_col].astype(str)

    row["recomputed_accuracy"] = accuracy_score(y_true, y_pred)
    row["recomputed_balanced_accuracy"] = balanced_accuracy_score(
        y_true, y_pred
    )
    row["recomputed_macro_f1"] = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )[2]

    saved = metric_df.iloc[0]
    discrepancies = []

    for metric in ["accuracy", "balanced_accuracy", "macro_f1"]:
        saved_value = pd.to_numeric(
            pd.Series([saved.get(metric, np.nan)]), errors="coerce"
        ).iloc[0]
        calculated = row[f"recomputed_{metric}"]

        row[f"saved_{metric}"] = saved_value

        if pd.isna(saved_value):
            discrepancies.append(f"{metric}: missing from saved metrics")
        elif not np.isclose(calculated, saved_value, atol=1e-5):
            discrepancies.append(
                f"{metric}: recomputed={calculated:.6f}, saved={saved_value:.6f}"
            )

    row["status"] = (
        "PASS: metrics reproduced"
        if not discrepancies
        else "REVIEW: discrepancy"
    )
    row["details"] = "; ".join(discrepancies)
    results.append(row)

result_df = pd.DataFrame(results)
report_path = output_dir / "prediction_metric_reproduction_check.csv"
result_df.to_csv(report_path, index=False)

display(result_df)
print("Saved refreshed report:", report_path)


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,model,prediction_rows,recomputed_accuracy,saved_accuracy,recomputed_balanced_accuracy,saved_balanced_accuracy,recomputed_macro_f1,saved_macro_f1,status,details
0,Lightweight Mel CNN,1561,0.716848,0.716848,0.310168,0.310168,0.212880,0.212880,PASS: metrics reproduced,
1,CNN-LSTM,1561,0.813581,0.813581,0.226085,0.226085,0.167396,0.146471,REVIEW: discrepancy,"macro_f1: recomputed=0.167396, saved=0.146471"
2,Multi-Feature CNN,1561,0.855221,0.855221,0.166417,0.166417,0.153660,0.153660,PASS: metrics reproduced,
3,Regularized Multi-Feature CNN (12B),1561,0.856502,0.856502,0.166667,0.166667,0.153784,0.153784,PASS: metrics reproduced,


Saved refreshed report: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/prediction_metric_reproduction_check.csv


In [25]:

from pathlib import Path
import pandas as pd
from sklearn.metrics import classification_report

PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
).resolve()

pred_path = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "cnn_lstm" / "cnn_lstm_test_predictions.csv"
)

df = pd.read_csv(pred_path)

y_true = df["diagnosis"].astype(str)
y_pred = df["predicted_diagnosis"].astype(str)

print("True diagnosis counts:")
display(y_true.value_counts().rename_axis("diagnosis").to_frame("count"))

print("\nPredicted diagnosis counts:")
display(y_pred.value_counts().rename_axis("diagnosis").to_frame("count"))

print("\nPer-class report:")
report = classification_report(
    y_true,
    y_pred,
    output_dict=True,
    zero_division=0
)
report_df = pd.DataFrame(report).T
display(report_df)

print("\nMacro F1 using labels present in the true labels:")
true_classes = sorted(y_true.unique())
report_true_classes = classification_report(
    y_true,
    y_pred,
    labels=true_classes,
    output_dict=True,
    zero_division=0
)
print("Macro F1:", report_true_classes["macro avg"]["f1-score"])

print("\nMacro F1 using all labels present in either true or predicted labels:")
all_classes = sorted(set(y_true) | set(y_pred))
report_all_classes = classification_report(
    y_true,
    y_pred,
    labels=all_classes,
    output_dict=True,
    zero_division=0
)
print("Macro F1:", report_all_classes["macro avg"]["f1-score"])


True diagnosis counts:


,count
diagnosis,
COPD,1337
Healthy,84
Pneumonia,52
URTI,49
Bronchiectasis,20
Bronchiolitis,19



Predicted diagnosis counts:


,count
diagnosis,
COPD,1411
Pneumonia,115
Asthma,35



Per-class report:


,precision,recall,f1-score,support
Asthma,0.000000,0.000000,0.000000,0.000000
Bronchiectasis,0.000000,0.000000,0.000000,20.000000
Bronchiolitis,0.000000,0.000000,0.000000,19.000000
COPD,0.884479,0.933433,0.908297,1337.000000
Healthy,0.000000,0.000000,0.000000,84.000000
Pneumonia,0.191304,0.423077,0.263473,52.000000
URTI,0.000000,0.000000,0.000000,49.000000
accuracy,0.813581,0.813581,0.813581,0.813581
macro avg,0.153683,0.193787,0.167396,1561.000000
weighted avg,0.763931,0.813581,0.786735,1561.000000



Macro F1 using labels present in the true labels:
Macro F1: 0.19529499952060944

Macro F1 using all labels present in either true or predicted labels:
Macro F1: 0.1673957138748081


In [10]:

from pathlib import Path
import pandas as pd
import hashlib

# Resolve project root
project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

output_dir = project_root / "reports" / "final_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

# Directories containing project outputs
scan_roots = [
    project_root / "models",
    project_root / "data" / "processed",
    project_root / "reports",
    project_root / "figures",
    project_root / "outputs",
]

# Avoid scanning the final inventory output repeatedly.
excluded_parts = {".git", ".venv", "__pycache__", ".ipynb_checkpoints"}

records = []

for root in scan_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        relative_path = path.relative_to(project_root)

        if any(part in excluded_parts for part in relative_path.parts):
            continue

        # Avoid including generated inventory files in their own inventory.
        if output_dir in path.parents:
            continue

        suffix = path.suffix.lower()
        name = path.name.lower()

        if suffix in {".pt", ".pth", ".onnx"}:
            category = "model_checkpoint"
        elif suffix in {".csv", ".parquet"}:
            category = "tabular_results"
        elif suffix in {".png", ".jpg", ".jpeg", ".svg", ".pdf"}:
            category = "figure_or_report"
        elif suffix in {".json", ".yaml", ".yml"}:
            category = "configuration_or_metadata"
        elif "history" in name:
            category = "training_history"
        else:
            category = "other_artifact"

        try:
            stat = path.stat()
            records.append({
                "relative_path": str(relative_path),
                "filename": path.name,
                "category": category,
                "extension": suffix,
                "size_bytes": stat.st_size,
                "size_kb": round(stat.st_size / 1024, 2),
                "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                "exists": True,
            })
        except OSError as exc:
            records.append({
                "relative_path": str(relative_path),
                "filename": path.name,
                "category": category,
                "extension": suffix,
                "size_bytes": None,
                "size_kb": None,
                "sha256": None,
                "exists": False,
                "error": str(exc),
            })

inventory_df = pd.DataFrame(records)

inventory_path = output_dir / "artifact_inventory.csv"
inventory_df.to_csv(inventory_path, index=False)

print("Project root:", project_root)
print("Artifacts indexed:", len(inventory_df))
print("\nArtifacts by category:")

if not inventory_df.empty:
    display(
        inventory_df.groupby("category")
        .agg(
            artifact_count=("relative_path", "count"),
            total_size_kb=("size_kb", "sum"),
        )
        .reset_index()
    )

    print("\nModel checkpoints:")
    display(
        inventory_df[
            inventory_df["category"] == "model_checkpoint"
        ][["relative_path", "size_kb", "sha256"]]
    )

print("\nInventory saved to:", inventory_path)


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Artifacts indexed: 7027

Artifacts by category:


,category,artifact_count,total_size_kb
0,configuration_or_metadata,11,5.47
1,figure_or_report,10,824.36
2,model_checkpoint,4,1576.60
3,other_artifact,6916,145853.88
4,tabular_results,86,12167.85



Model checkpoints:


,relative_path,size_kb,sha256
6960,data/processed/deep_learning/cnn_lstm/best_cnn...,364.66,b9857e691d87c8c6be563002838f66346da2ef9a8ba6cc...
6971,data/processed/deep_learning/cnn_baseline/best...,303.65,a4ab3323f2db471810cfbee86cc3f490db58ee2bb62797...
6988,data/processed/deep_learning/multifeature_cnn_...,454.64,dc62347524cca4deaec06b7c5d993889db3dc04ca949dc...
7005,data/processed/deep_learning/multifeature_cnn/...,453.65,dc0d51c55c48bb222b54c10a5f520bd4ca27c7d7483bcd...



Inventory saved to: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/artifact_inventory.csv


In [11]:

from pathlib import Path
import pandas as pd
import json

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

dl_root = project_root / "data" / "processed" / "deep_learning"
output_dir = project_root / "reports" / "final_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

model_specs = [
    {
        "model_id": "cnn_baseline",
        "model_name": "Lightweight Mel CNN",
        "architecture": "LightweightMelCNN",
        "checkpoint": "cnn_baseline/best_lightweight_mel_cnn.pt",
        "predictions": "cnn_baseline/cnn_test_predictions.csv",
        "metrics_candidates": [
            "cnn_baseline/cnn_baseline_metrics.csv",
            "cnn_baseline/metrics.csv",
        ],
        "history": "cnn_baseline/cnn_training_history.csv",
        "config": "cnn_baseline/cnn_training_config.json",
        "features": "Log-Mel",
    },
    {
        "model_id": "cnn_lstm",
        "model_name": "CNN-LSTM",
        "architecture": "CNNLSTM",
        "checkpoint": "cnn_lstm/best_cnn_lstm.pt",
        "predictions": "cnn_lstm/cnn_lstm_test_predictions.csv",
        "metrics_candidates": [
            "cnn_lstm/cnn_lstm_metrics.csv",
            "cnn_lstm/metrics.csv",
        ],
        "history": "cnn_lstm/cnn_lstm_training_history.csv",
        "config": "cnn_lstm/cnn_lstm_training_config.json",
        "features": "Log-Mel",
    },
    {
        "model_id": "multifeature_cnn",
        "model_name": "Multi-Feature CNN",
        "architecture": "MultiFeatureCNN",
        "checkpoint": "multifeature_cnn/best_multifeature_cnn.pt",
        "predictions": "multifeature_cnn/multifeature_cnn_test_predictions.csv",
        "metrics_candidates": [
            "multifeature_cnn/multifeature_cnn_metrics.csv",
            "multifeature_cnn/metrics.csv",
        ],
        "history": "multifeature_cnn/multifeature_cnn_training_history.csv",
        "config": "multifeature_cnn/multifeature_cnn_training_config.json",
        "features": "Log-Mel, MFCC, Chroma",
    },
    {
        "model_id": "multifeature_cnn_12b",
        "model_name": "Regularized Multi-Feature CNN",
        "architecture": "RegularizedMultiFeatureCNN",
        "checkpoint": "multifeature_cnn_12b/best_regularized_multifeature_cnn.pt",
        "predictions": "multifeature_cnn_12b/test_predictions.csv",
        "metrics_candidates": [
            "multifeature_cnn_12b/metrics.csv",
            "multifeature_cnn_12b/multifeature_cnn_12b_metrics.csv",
        ],
        "history": "multifeature_cnn_12b/training_history.csv",
        "config": "multifeature_cnn_12b/training_config.json",
        "features": "Log-Mel, MFCC, Chroma",
    },
]

registry_rows = []

for spec in model_specs:
    checkpoint = dl_root / spec["checkpoint"]
    prediction_file = dl_root / spec["predictions"]
    history_file = dl_root / spec["history"]
    config_file = dl_root / spec["config"]

    metrics_file = next(
        (
            dl_root / candidate
            for candidate in spec["metrics_candidates"]
            if (dl_root / candidate).exists()
        ),
        None,
    )

    # Checkpoint existence and prediction/history/config availability
    row = {
        "model_id": spec["model_id"],
        "model_name": spec["model_name"],
        "architecture": spec["architecture"],
        "features": spec["features"],
        "checkpoint_path": str(checkpoint.relative_to(project_root)),
        "checkpoint_exists": checkpoint.exists(),
        "prediction_path": str(prediction_file.relative_to(project_root)),
        "prediction_exists": prediction_file.exists(),
        "metrics_path": (
            str(metrics_file.relative_to(project_root))
            if metrics_file else None
        ),
        "metrics_exists": metrics_file is not None,
        "history_path": str(history_file.relative_to(project_root)),
        "history_exists": history_file.exists(),
        "config_path": str(config_file.relative_to(project_root)),
        "config_exists": config_file.exists(),
        "normalization_path": None,
        "normalization_exists": False,
    }

    # Attach the known normalization artifact, where applicable.
    if spec["model_id"] in ("cnn_baseline", "cnn_lstm"):
        norm_path = dl_root / "logmel_train_normalization.csv"
    elif spec["model_id"] == "multifeature_cnn":
        norm_path = (
            dl_root / "multifeature_cnn" /
            "multifeature_train_normalization.csv"
        )
    else:
        # Do not assume the regularized model uses another model's
        # normalization statistics without checking its training notebook.
        norm_path = None

    if norm_path is not None:
        row["normalization_path"] = str(norm_path.relative_to(project_root))
        row["normalization_exists"] = norm_path.exists()

    registry_rows.append(row)

registry_df = pd.DataFrame(registry_rows)

registry_path = output_dir / "model_registry.csv"
registry_df.to_csv(registry_path, index=False)

display(registry_df)

print("\nSaved model registry:", registry_path)
print("\nMissing artifact counts:")
display(
    registry_df[
        [
            "model_name",
            "checkpoint_exists",
            "prediction_exists",
            "metrics_exists",
            "history_exists",
            "config_exists",
            "normalization_exists",
        ]
    ]
)


,model_id,model_name,architecture,features,checkpoint_path,checkpoint_exists,prediction_path,prediction_exists,metrics_path,metrics_exists,history_path,history_exists,config_path,config_exists,normalization_path,normalization_exists
0,cnn_baseline,Lightweight Mel CNN,LightweightMelCNN,Log-Mel,data/processed/deep_learning/cnn_baseline/best...,True,data/processed/deep_learning/cnn_baseline/cnn_...,True,data/processed/deep_learning/cnn_baseline/cnn_...,True,data/processed/deep_learning/cnn_baseline/cnn_...,True,data/processed/deep_learning/cnn_baseline/cnn_...,True,data/processed/deep_learning/logmel_train_norm...,True
1,cnn_lstm,CNN-LSTM,CNNLSTM,Log-Mel,data/processed/deep_learning/cnn_lstm/best_cnn...,True,data/processed/deep_learning/cnn_lstm/cnn_lstm...,True,data/processed/deep_learning/cnn_lstm/cnn_lstm...,True,data/processed/deep_learning/cnn_lstm/cnn_lstm...,True,data/processed/deep_learning/cnn_lstm/cnn_lstm...,True,data/processed/deep_learning/logmel_train_norm...,True
2,multifeature_cnn,Multi-Feature CNN,MultiFeatureCNN,"Log-Mel, MFCC, Chroma",data/processed/deep_learning/multifeature_cnn/...,True,data/processed/deep_learning/multifeature_cnn/...,True,data/processed/deep_learning/multifeature_cnn/...,True,data/processed/deep_learning/multifeature_cnn/...,True,data/processed/deep_learning/multifeature_cnn/...,True,data/processed/deep_learning/multifeature_cnn/...,True
3,multifeature_cnn_12b,Regularized Multi-Feature CNN,RegularizedMultiFeatureCNN,"Log-Mel, MFCC, Chroma",data/processed/deep_learning/multifeature_cnn_...,True,data/processed/deep_learning/multifeature_cnn_...,True,data/processed/deep_learning/multifeature_cnn_...,True,data/processed/deep_learning/multifeature_cnn_...,True,data/processed/deep_learning/multifeature_cnn_...,True,NaN,False



Saved model registry: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/model_registry.csv

Missing artifact counts:


,model_name,checkpoint_exists,prediction_exists,metrics_exists,history_exists,config_exists,normalization_exists
0,Lightweight Mel CNN,True,True,True,True,True,True
1,CNN-LSTM,True,True,True,True,True,True
2,Multi-Feature CNN,True,True,True,True,True,True
3,Regularized Multi-Feature CNN,True,True,True,True,True,False


In [12]:

from pathlib import Path
import shutil
import pandas as pd
import json

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

dl_root = project_root / "data" / "processed" / "deep_learning"
package_root = project_root / "models" / "saved_models"
package_root.mkdir(parents=True, exist_ok=True)

model_specs = [
    {
        "model_id": "lightweight_mel_cnn",
        "checkpoint": dl_root / "cnn_baseline" / "best_lightweight_mel_cnn.pt",
        "config": dl_root / "cnn_baseline" / "cnn_training_config.json",
        "predictions": dl_root / "cnn_baseline" / "cnn_test_predictions.csv",
        "history": dl_root / "cnn_baseline" / "cnn_training_history.csv",
        "normalization": dl_root / "logmel_train_normalization.csv",
    },
    {
        "model_id": "cnn_lstm",
        "checkpoint": dl_root / "cnn_lstm" / "best_cnn_lstm.pt",
        "config": dl_root / "cnn_lstm" / "cnn_lstm_training_config.json",
        "predictions": dl_root / "cnn_lstm" / "cnn_lstm_test_predictions.csv",
        "history": dl_root / "cnn_lstm" / "cnn_lstm_training_history.csv",
        "normalization": dl_root / "logmel_train_normalization.csv",
    },
    {
        "model_id": "multifeature_cnn",
        "checkpoint": dl_root / "multifeature_cnn" / "best_multifeature_cnn.pt",
        "config": dl_root / "multifeature_cnn" / "multifeature_cnn_training_config.json",
        "predictions": dl_root / "multifeature_cnn" / "multifeature_cnn_test_predictions.csv",
        "history": dl_root / "multifeature_cnn" / "multifeature_cnn_training_history.csv",
        "normalization": (
            dl_root / "multifeature_cnn" / "multifeature_train_normalization.csv"
        ),
    },
    {
        "model_id": "regularized_multifeature_cnn",
        "checkpoint": (
            dl_root / "multifeature_cnn_12b" /
            "best_regularized_multifeature_cnn.pt"
        ),
        "config": dl_root / "multifeature_cnn_12b" / "training_config.json",
        "predictions": dl_root / "multifeature_cnn_12b" / "test_predictions.csv",
        "history": dl_root / "multifeature_cnn_12b" / "training_history.csv",
        "normalization": None,
    },
]

copy_records = []

for spec in model_specs:
    model_dir = package_root / spec["model_id"]
    model_dir.mkdir(parents=True, exist_ok=True)

    for role in ("checkpoint", "config", "predictions", "history", "normalization"):
        source = spec.get(role)

        if source is None:
            copy_records.append({
                "model_id": spec["model_id"],
                "artifact": role,
                "source": None,
                "destination": None,
                "status": "NOT_SPECIFIED",
            })
            continue

        if not source.exists():
            copy_records.append({
                "model_id": spec["model_id"],
                "artifact": role,
                "source": str(source),
                "destination": None,
                "status": "SOURCE_MISSING",
            })
            continue

        destination = model_dir / source.name

        # Avoid needless copying when source and destination are identical.
        if source.resolve() != destination.resolve():
            shutil.copy2(source, destination)

        copy_records.append({
            "model_id": spec["model_id"],
            "artifact": role,
            "source": str(source.relative_to(project_root)),
            "destination": str(destination.relative_to(project_root)),
            "status": "COPIED_OR_ALREADY_PRESENT",
        })

copy_df = pd.DataFrame(copy_records)

copy_log = project_root / "reports" / "final_artifacts" / "model_package_copy_log.csv"
copy_log.parent.mkdir(parents=True, exist_ok=True)
copy_df.to_csv(copy_log, index=False)

print("Package directory:", package_root)
display(copy_df)
print("\nCopy log saved to:", copy_log)


Package directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/saved_models


,model_id,artifact,source,destination,status
0,lightweight_mel_cnn,checkpoint,data/processed/deep_learning/cnn_baseline/best...,models/saved_models/lightweight_mel_cnn/best_l...,COPIED_OR_ALREADY_PRESENT
1,lightweight_mel_cnn,config,data/processed/deep_learning/cnn_baseline/cnn_...,models/saved_models/lightweight_mel_cnn/cnn_tr...,COPIED_OR_ALREADY_PRESENT
2,lightweight_mel_cnn,predictions,data/processed/deep_learning/cnn_baseline/cnn_...,models/saved_models/lightweight_mel_cnn/cnn_te...,COPIED_OR_ALREADY_PRESENT
3,lightweight_mel_cnn,history,data/processed/deep_learning/cnn_baseline/cnn_...,models/saved_models/lightweight_mel_cnn/cnn_tr...,COPIED_OR_ALREADY_PRESENT
4,lightweight_mel_cnn,normalization,data/processed/deep_learning/logmel_train_norm...,models/saved_models/lightweight_mel_cnn/logmel...,COPIED_OR_ALREADY_PRESENT
5,cnn_lstm,checkpoint,data/processed/deep_learning/cnn_lstm/best_cnn...,models/saved_models/cnn_lstm/best_cnn_lstm.pt,COPIED_OR_ALREADY_PRESENT
6,cnn_lstm,config,data/processed/deep_learning/cnn_lstm/cnn_lstm...,models/saved_models/cnn_lstm/cnn_lstm_training...,COPIED_OR_ALREADY_PRESENT
7,cnn_lstm,predictions,data/processed/deep_learning/cnn_lstm/cnn_lstm...,models/saved_models/cnn_lstm/cnn_lstm_test_pre...,COPIED_OR_ALREADY_PRESENT
8,cnn_lstm,history,data/processed/deep_learning/cnn_lstm/cnn_lstm...,models/saved_models/cnn_lstm/cnn_lstm_training...,COPIED_OR_ALREADY_PRESENT
9,cnn_lstm,normalization,data/processed/deep_learning/logmel_train_norm...,models/saved_models/cnn_lstm/logmel_train_norm...,COPIED_OR_ALREADY_PRESENT



Copy log saved to: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/model_package_copy_log.csv


In [13]:

from pathlib import Path
import pandas as pd
import hashlib
import json

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

package_root = project_root / "models" / "saved_models"
output_dir = project_root / "reports" / "final_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

manifest_rows = []

required_roles = {
    "checkpoint": [".pt", ".pth"],
    "config": [".json"],
    "normalization": [".csv", ".json"],
    "predictions": [".csv"],
    "history": [".csv"],
}

if not package_root.exists():
    raise FileNotFoundError(
        f"Package directory not found: {package_root}. "
        "Run the previous packaging cell first."
    )

for model_dir in sorted(p for p in package_root.iterdir() if p.is_dir()):
    files = [p for p in model_dir.iterdir() if p.is_file()]

    for path in files:
        manifest_rows.append({
            "model_id": model_dir.name,
            "artifact": path.name,
            "relative_path": str(path.relative_to(project_root)),
            "size_bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            "status": "PRESENT",
        })

    # Record expected roles without pretending that any missing role exists.
    for role, extensions in required_roles.items():
        matches = [
            p for p in files
            if p.suffix.lower() in extensions
            and (
                role == "checkpoint"
                or role == "config" and p.name.lower().endswith(".json")
                or role == "normalization" and "normal" in p.name.lower()
                or role == "predictions" and "prediction" in p.name.lower()
                or role == "history" and "history" in p.name.lower()
            )
        ]

        if not matches:
            manifest_rows.append({
                "model_id": model_dir.name,
                "artifact": role,
                "relative_path": None,
                "size_bytes": None,
                "sha256": None,
                "status": "MISSING_OR_NOT_IDENTIFIED",
            })

manifest_df = pd.DataFrame(manifest_rows)

manifest_csv = output_dir / "model_package_manifest.csv"
manifest_df.to_csv(manifest_csv, index=False)

# JSON companion for programmatic use.
manifest_json = output_dir / "model_package_manifest.json"
manifest_json.write_text(
    json.dumps(manifest_rows, indent=2),
    encoding="utf-8"
)

print("Package root:", package_root)
print("Manifest CSV:", manifest_csv)
print("Manifest JSON:", manifest_json)

print("\nManifest summary:")
if not manifest_df.empty:
    display(
        manifest_df.groupby(["model_id", "status"])
        .size()
        .reset_index(name="artifact_count")
    )

    print("\nFull manifest:")
    display(manifest_df)


Package root: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/saved_models
Manifest CSV: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/model_package_manifest.csv
Manifest JSON: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/model_package_manifest.json

Manifest summary:


,model_id,status,artifact_count
0,cnn_lstm,PRESENT,5
1,lightweight_mel_cnn,PRESENT,5
2,multifeature_cnn,PRESENT,5
3,regularized_multifeature_cnn,MISSING_OR_NOT_IDENTIFIED,1
4,regularized_multifeature_cnn,PRESENT,4



Full manifest:


,model_id,artifact,relative_path,size_bytes,sha256,status
0,cnn_lstm,logmel_train_normalization.csv,models/saved_models/cnn_lstm/logmel_train_norm...,62.0,8704674c3456062030052cab13cb362f061a783efec4ab...,PRESENT
1,cnn_lstm,cnn_lstm_training_history.csv,models/saved_models/cnn_lstm/cnn_lstm_training...,1335.0,3155e5b21cf8737fc05431ec57164d50b52f25a3179899...,PRESENT
2,cnn_lstm,best_cnn_lstm.pt,models/saved_models/cnn_lstm/best_cnn_lstm.pt,373412.0,b9857e691d87c8c6be563002838f66346da2ef9a8ba6cc...,PRESENT
3,cnn_lstm,cnn_lstm_training_config.json,models/saved_models/cnn_lstm/cnn_lstm_training...,682.0,475dbef065fa1e926fd84a4d4e11581ed29a6d5393f86a...,PRESENT
4,cnn_lstm,cnn_lstm_test_predictions.csv,models/saved_models/cnn_lstm/cnn_lstm_test_pre...,210365.0,81fa11385d52f773e0c33996fdaa8f695a2cc111d92908...,PRESENT
5,lightweight_mel_cnn,logmel_train_normalization.csv,models/saved_models/lightweight_mel_cnn/logmel...,62.0,8704674c3456062030052cab13cb362f061a783efec4ab...,PRESENT
6,lightweight_mel_cnn,cnn_test_predictions.csv,models/saved_models/lightweight_mel_cnn/cnn_te...,223653.0,9f8434e2c56cb326cf46fa61fb9964eaae480ca32b507e...,PRESENT
7,lightweight_mel_cnn,cnn_training_config.json,models/saved_models/lightweight_mel_cnn/cnn_tr...,459.0,262022167bf08f6c1f26b6c113923bcd4373824af5f3c3...,PRESENT
8,lightweight_mel_cnn,best_lightweight_mel_cnn.pt,models/saved_models/lightweight_mel_cnn/best_l...,310935.0,a4ab3323f2db471810cfbee86cc3f490db58ee2bb62797...,PRESENT
9,lightweight_mel_cnn,cnn_training_history.csv,models/saved_models/lightweight_mel_cnn/cnn_tr...,3380.0,075181a6b7e8640681ece640391f40689f76fbe163d937...,PRESENT


In [14]:

from pathlib import Path
import pandas as pd
from datetime import datetime

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

output_dir = project_root / "reports" / "final_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

search_roots = [
    project_root / "data" / "processed" / "deep_learning",
    project_root / "data" / "processed" / "classical_ml",
    output_dir,
]

# Collect saved metric and comparison CSVs.
metric_files = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*.csv"):
        name = path.name.lower()

        if (
            any(term in name for term in ("metric", "comparison"))
            and path.name != "final_results_source_inventory.csv"
        ):
            metric_files.append(path)

metric_files = sorted(set(metric_files))

inventory_rows = []
report_sections = []

report_sections.append("# Respiratory Disease Classification — Final Results")
report_sections.append("")
report_sections.append(
    f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
)
report_sections.append("")
report_sections.append(
    "This report consolidates existing metric and comparison CSV files. "
    "Values are taken from the saved artifacts and are not retrained or "
    "recomputed by this report-generation step."
)
report_sections.append("")

for path in metric_files:
    relative_path = str(path.relative_to(project_root))

    try:
        df = pd.read_csv(path)

        inventory_rows.append({
            "source_file": relative_path,
            "rows": len(df),
            "columns": len(df.columns),
            "column_names": ", ".join(map(str, df.columns)),
            "read_status": "READABLE",
        })

        report_sections.append("---")
        report_sections.append("")
        report_sections.append(f"## {path.name}")
        report_sections.append("")
        report_sections.append(f"Source: `{relative_path}`")
        report_sections.append("")
        report_sections.append(f"Rows: {len(df)}")
        report_sections.append("")
        report_sections.append(df.to_markdown(index=False))
        report_sections.append("")

    except Exception as exc:
        inventory_rows.append({
            "source_file": relative_path,
            "rows": None,
            "columns": None,
            "column_names": None,
            "read_status": f"ERROR: {type(exc).__name__}: {exc}",
        })

report_sections.append("---")
report_sections.append("")
report_sections.append("## Evaluation and interpretation notes")
report_sections.append("")
report_sections.append(
    "- The general deep-learning test manifest contains 2,756 rows."
)
report_sections.append(
    "- Each of the four saved deep-learning prediction files contains "
    "1,561 rows."
)
report_sections.append(
    "- These sample counts differ. Do not treat model metrics as a direct "
    "head-to-head comparison until the exact evaluation splits are verified."
)
report_sections.append(
    "- Cycle-level and patient-level metrics represent different evaluation "
    "units and must remain separate."
)
report_sections.append(
    "- High accuracy alone does not establish good minority-class performance; "
    "review balanced accuracy, macro F1, and per-class results where available."
)
report_sections.append(
    "- This report consolidates available CSVs. It does not independently "
    "verify that every metric was calculated correctly."
)

report_path = output_dir / "final_results_report.md"
report_path.write_text(
    "\n".join(report_sections),
    encoding="utf-8"
)

inventory_df = pd.DataFrame(inventory_rows)
inventory_path = output_dir / "final_results_source_inventory.csv"
inventory_df.to_csv(inventory_path, index=False)

print("Metric/comparison files discovered:", len(metric_files))
print("Results report:", report_path)
print("Source inventory:", inventory_path)

if not inventory_df.empty:
    display(inventory_df)
else:
    print(
        "No metric/comparison CSV files were found. "
        "Check the output directory and filenames."
    )


Metric/comparison files discovered: 21
Results report: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/final_results_report.md
Source inventory: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/final_results_source_inventory.csv


,source_file,rows,columns,column_names,read_status
0,data/processed/classical_ml/.ipynb_checkpoints...,3.0,8.0,"model, accuracy, balanced_accuracy, macro_prec...",READABLE
1,data/processed/classical_ml/.ipynb_checkpoints...,NaN,NaN,NaN,ERROR: ImportError: `Import tabulate` failed. ...
2,data/processed/classical_ml/classical_ml_model...,3.0,8.0,"model, accuracy, balanced_accuracy, macro_prec...",READABLE
3,data/processed/classical_ml/classical_ml_model...,NaN,NaN,NaN,ERROR: ImportError: `Import tabulate` failed. ...
4,data/processed/classical_ml/final_classical_ml...,3.0,8.0,"Model, Accuracy, Balanced Accuracy, Macro Prec...",READABLE
5,data/processed/classical_ml/final_classical_ml...,NaN,NaN,NaN,ERROR: ImportError: `Import tabulate` failed. ...
6,data/processed/deep_learning/cnn_baseline/.ipy...,1.0,11.0,"model, accuracy, balanced_accuracy, macro_prec...",READABLE
7,data/processed/deep_learning/cnn_baseline/.ipy...,NaN,NaN,NaN,ERROR: ImportError: `Import tabulate` failed. ...
8,data/processed/deep_learning/cnn_baseline/cnn_...,1.0,11.0,"model, accuracy, balanced_accuracy, macro_prec...",READABLE
9,data/processed/deep_learning/cnn_baseline/cnn_...,NaN,NaN,NaN,ERROR: ImportError: `Import tabulate` failed. ...


In [15]:

from pathlib import Path
import pandas as pd
from datetime import datetime
import math

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

output_dir = project_root / "reports" / "final_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

search_roots = [
    project_root / "data" / "processed" / "deep_learning",
    project_root / "data" / "processed" / "classical_ml",
    output_dir,
]

def markdown_table(df):
    """Render a DataFrame as Markdown without requiring tabulate."""
    if df.empty:
        return "_No rows available._"

    def clean(value):
        if pd.isna(value):
            return ""
        value = str(value).replace("|", "\\|")
        return value.replace("\n", " ")

    headers = [clean(c) for c in df.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]

    for row in df.itertuples(index=False, name=None):
        lines.append("| " + " | ".join(clean(v) for v in row) + " |")

    return "\n".join(lines)

metric_files = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*.csv"):
        if ".ipynb_checkpoints" in path.parts:
            continue

        if path.name in {
            "final_results_source_inventory.csv",
        }:
            continue

        # Include actual metric/comparison files, not every CSV.
        name = path.name.lower()
        if any(term in name for term in ("metric", "comparison")):
            metric_files.append(path)

metric_files = sorted(set(metric_files))

sections = [
    "# Respiratory Disease Classification — Final Results",
    "",
    f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    "This report consolidates existing metric and comparison CSV files. "
    "It does not retrain models or independently recalculate all metrics.",
    "",
]

inventory = []

for path in metric_files:
    relative_path = str(path.relative_to(project_root))

    try:
        df = pd.read_csv(path)

        inventory.append({
            "source_file": relative_path,
            "rows": len(df),
            "columns": len(df.columns),
            "read_status": "READABLE",
        })

        sections.extend([
            "---",
            "",
            f"## {path.name}",
            "",
            f"Source: `{relative_path}`",
            "",
            markdown_table(df),
            "",
        ])

    except Exception as exc:
        inventory.append({
            "source_file": relative_path,
            "rows": None,
            "columns": None,
            "read_status": f"ERROR: {type(exc).__name__}: {exc}",
        })

sections.extend([
    "---",
    "",
    "## Evaluation notes",
    "",
    "- The general deep-learning test manifest contains 2,756 rows, "
    "while each of the four saved deep-learning prediction files contains "
    "1,561 rows. Verify evaluation-split provenance before comparing models.",
    "- Cycle-level and patient-level results use different evaluation units "
    "and must be reported separately.",
    "- Accuracy should be interpreted alongside balanced accuracy and macro F1 "
    "because class imbalance may affect the results.",
    "- This report preserves existing saved results; it does not prove that "
    "all models used identical test samples.",
    "",
])

report_path = output_dir / "final_results_report.md"
report_path.write_text("\n".join(sections), encoding="utf-8")

inventory_path = output_dir / "final_results_source_inventory.csv"
pd.DataFrame(inventory).to_csv(inventory_path, index=False)

print("Metric/comparison files included:", len(metric_files))
print("Unreadable source files:", sum(
    row["read_status"] != "READABLE" for row in inventory
))
print("Report saved:", report_path)
print("Inventory saved:", inventory_path)


Metric/comparison files included: 18
Unreadable source files: 0
Report saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/final_results_report.md
Inventory saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/final_results_source_inventory.csv


In [16]:

from pathlib import Path
import pandas as pd
import json

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

artifact_dir = project_root / "reports" / "final_artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

checks = []

def add_check(category, item, path, requirement="Required"):
    path = Path(path)
    exists = path.exists()

    if exists and path.is_file():
        status = "PASS"
        details = f"File exists ({path.stat().st_size:,} bytes)"
    elif exists and path.is_dir():
        status = "PASS"
        details = "Directory exists"
    else:
        status = "MISSING"
        details = "Path does not exist"

    checks.append({
        "category": category,
        "item": item,
        "path": str(path.relative_to(project_root))
            if path.is_relative_to(project_root) else str(path),
        "requirement": requirement,
        "status": status,
        "details": details,
    })

# Four saved deep-learning models
models = {
    "Lightweight Mel CNN": (
        "models/saved_models/lightweight_mel_cnn",
        "best_lightweight_mel_cnn.pt",
    ),
    "CNN-LSTM": (
        "models/saved_models/cnn_lstm",
        "best_cnn_lstm.pt",
    ),
    "Multi-Feature CNN": (
        "models/saved_models/multifeature_cnn",
        "best_multifeature_cnn.pt",
    ),
    "Regularized Multi-Feature CNN": (
        "models/saved_models/regularized_multifeature_cnn",
        "best_regularized_multifeature_cnn.pt",
    ),
}

for model_name, (folder, checkpoint_name) in models.items():
    package_dir = project_root / folder
    add_check("Model package", f"{model_name} package directory", package_dir)

    add_check(
        "Model checkpoint",
        f"{model_name} checkpoint",
        package_dir / checkpoint_name,
    )

# Core final reports
for filename in [
    "final_results_report.md",
    "final_results_source_inventory.csv",
    "cycle_level_model_comparison.csv",
    "patient_level_model_comparison.csv",
    "model_registry.csv",
    "model_package_manifest.csv",
    "model_package_manifest.json",
    "artifact_inventory.csv",
    "architecture_inference_validation.csv",
    "prediction_validation.csv",
    "checkpoint_integrity_check.csv",
]:
    add_check(
        "Final report",
        filename,
        artifact_dir / filename,
    )

# Project source and test files
add_check(
    "Notebook",
    "Notebook 17",
    project_root / "notebooks" /
    "17_Final_Artifact_Generation_and_Validation.ipynb",
)

add_check(
    "Tests",
    "Project tests directory",
    project_root / "tests",
)

# Save validation results
validation_df = pd.DataFrame(checks)
validation_csv = artifact_dir / "final_project_validation.csv"
validation_json = artifact_dir / "final_project_validation.json"

validation_df.to_csv(validation_csv, index=False)
validation_json.write_text(
    json.dumps(checks, indent=2),
    encoding="utf-8",
)

print("FINAL PROJECT VALIDATION")
print("=" * 60)
print("Total checks:", len(validation_df))
print("PASS:", int((validation_df["status"] == "PASS").sum()))
print("MISSING:", int((validation_df["status"] == "MISSING").sum()))
print()
print("Validation CSV:", validation_csv)
print("Validation JSON:", validation_json)

display(
    validation_df[
        ["category", "item", "status", "details"]
    ]
)


FINAL PROJECT VALIDATION
Total checks: 21
PASS: 21
MISSING: 0

Validation CSV: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/final_project_validation.csv
Validation JSON: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/final_project_validation.json


,category,item,status,details
0,Model package,Lightweight Mel CNN package directory,PASS,Directory exists
1,Model checkpoint,Lightweight Mel CNN checkpoint,PASS,"File exists (310,935 bytes)"
2,Model package,CNN-LSTM package directory,PASS,Directory exists
3,Model checkpoint,CNN-LSTM checkpoint,PASS,"File exists (373,412 bytes)"
4,Model package,Multi-Feature CNN package directory,PASS,Directory exists
5,Model checkpoint,Multi-Feature CNN checkpoint,PASS,"File exists (464,538 bytes)"
6,Model package,Regularized Multi-Feature CNN package directory,PASS,Directory exists
7,Model checkpoint,Regularized Multi-Feature CNN checkpoint,PASS,"File exists (465,550 bytes)"
8,Final report,final_results_report.md,PASS,"File exists (15,082 bytes)"
9,Final report,final_results_source_inventory.csv,PASS,"File exists (1,522 bytes)"


In [17]:

from pathlib import Path
import zipfile
import hashlib
import pandas as pd
from datetime import datetime

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

artifact_dir = project_root / "reports" / "final_artifacts"
package_root = project_root / "outputs" / "final_submission"
package_root.mkdir(parents=True, exist_ok=True)

zip_path = package_root / "respiratory_disease_classification_artifacts.zip"

# Include packaged models, reports, validation outputs, and final notebook.
include_dirs = [
    project_root / "models" / "saved_models",
    artifact_dir,
]

include_files = [
    project_root / "notebooks" /
    "17_Final_Artifact_Generation_and_Validation.ipynb",
]

excluded_parts = {
    ".git",
    ".venv",
    "__pycache__",
    ".ipynb_checkpoints",
}

files_to_package = set()

for root in include_dirs:
    if root.exists():
        for path in root.rglob("*"):
            if not path.is_file():
                continue
            if any(part in excluded_parts for part in path.parts):
                continue
            if path.resolve() == zip_path.resolve():
                continue
            files_to_package.add(path)

for path in include_files:
    if path.is_file():
        files_to_package.add(path)

files_to_package = sorted(files_to_package)

if not files_to_package:
    raise RuntimeError(
        "No files were found to package. Check the project paths."
    )

manifest_rows = []

with zipfile.ZipFile(
    zip_path, mode="w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in files_to_package:
        relative_path = path.relative_to(project_root)
        archive_name = str(relative_path)

        data = path.read_bytes()
        archive.writestr(archive_name, data)

        manifest_rows.append({
            "archive_path": archive_name,
            "size_bytes": len(data),
            "sha256": hashlib.sha256(data).hexdigest(),
        })

manifest_df = pd.DataFrame(manifest_rows)
manifest_csv = artifact_dir / "final_submission_zip_manifest.csv"
manifest_df.to_csv(manifest_csv, index=False)

# Verify the ZIP can be read and all packaged files pass CRC checks.
with zipfile.ZipFile(zip_path, "r") as archive:
    bad_member = archive.testzip()
    archive_names = set(archive.namelist())

if bad_member is not None:
    raise RuntimeError(f"ZIP integrity check failed: {bad_member}")

missing_members = [
    row["archive_path"]
    for row in manifest_rows
    if row["archive_path"] not in archive_names
]

if missing_members:
    raise RuntimeError(
        f"ZIP is missing expected files: {missing_members}"
    )

print("FINAL PACKAGE CREATED")
print("=" * 55)
print("ZIP path:", zip_path)
print("Files packaged:", len(manifest_rows))
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024 ** 2), 2))
print("ZIP integrity check: PASS")
print("ZIP manifest:", manifest_csv)
print()
print("Included top-level folders:")
for name in sorted({row["archive_path"].split("/")[0]
                    for row in manifest_rows}):
    print("-", name)


FINAL PACKAGE CREATED
ZIP path: /Users/vs/Sem 5/Respiratory-Disease-Classification/outputs/final_submission/respiratory_disease_classification_artifacts.zip
Files packaged: 42
ZIP size (MB): 1.91
ZIP integrity check: PASS
ZIP manifest: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/final_submission_zip_manifest.csv

Included top-level folders:
- models
- notebooks
- reports


In [18]:

from pathlib import Path
import zipfile
import pandas as pd

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

zip_path = (
    project_root / "outputs" / "final_submission" /
    "respiratory_disease_classification_artifacts.zip"
)

required_members = [
    "models/saved_models/lightweight_mel_cnn/"
    "best_lightweight_mel_cnn.pt",

    "models/saved_models/cnn_lstm/"
    "best_cnn_lstm.pt",

    "models/saved_models/multifeature_cnn/"
    "best_multifeature_cnn.pt",

    "models/saved_models/regularized_multifeature_cnn/"
    "best_regularized_multifeature_cnn.pt",

    "reports/final_artifacts/final_results_report.md",
    "reports/final_artifacts/final_project_validation.csv",
    "reports/final_artifacts/cycle_level_model_comparison.csv",
    "reports/final_artifacts/patient_level_model_comparison.csv",

    "notebooks/17_Final_Artifact_Generation_and_Validation.ipynb",
]

if not zip_path.is_file():
    print("ZIP NOT FOUND:", zip_path)
    print("Run Step 29 first.")
else:
    with zipfile.ZipFile(zip_path, "r") as archive:
        names = set(archive.namelist())
        bad_member = archive.testzip()

    results = []

    for member in required_members:
        results.append({
            "required_file": member,
            "included": member in names,
        })

    results_df = pd.DataFrame(results)

    print("ZIP:", zip_path)
    print("ZIP size (MB):", round(zip_path.stat().st_size / (1024 ** 2), 2))
    print("Total files in ZIP:", len(names))
    print("ZIP CRC integrity:", "PASS" if bad_member is None else f"FAIL: {bad_member}")
    print("Required files included:",
          int(results_df["included"].sum()), "/", len(results_df))
    print()

    display(results_df)

    if bad_member is None and results_df["included"].all():
        print("PACKAGE CONTENT CHECK: PASS")
    else:
        print("PACKAGE CONTENT CHECK: NEEDS ATTENTION")


ZIP: /Users/vs/Sem 5/Respiratory-Disease-Classification/outputs/final_submission/respiratory_disease_classification_artifacts.zip
ZIP size (MB): 1.91
Total files in ZIP: 42
ZIP CRC integrity: PASS
Required files included: 9 / 9



,required_file,included
0,models/saved_models/lightweight_mel_cnn/best_l...,True
1,models/saved_models/cnn_lstm/best_cnn_lstm.pt,True
2,models/saved_models/multifeature_cnn/best_mult...,True
3,models/saved_models/regularized_multifeature_c...,True
4,reports/final_artifacts/final_results_report.md,True
5,reports/final_artifacts/final_project_validati...,True
6,reports/final_artifacts/cycle_level_model_comp...,True
7,reports/final_artifacts/patient_level_model_co...,True
8,notebooks/17_Final_Artifact_Generation_and_Val...,True


PACKAGE CONTENT CHECK: PASS


In [19]:

from sklearn.base import BaseEstimator
import pandas as pd

candidate_models = []

for name, obj in list(globals().items()):
    try:
        if isinstance(obj, BaseEstimator):
            if hasattr(obj, "fit") and hasattr(obj, "predict"):
                is_fitted = hasattr(obj, "n_features_in_") or hasattr(
                    obj, "estimators_"
                ) or hasattr(obj, "support_vectors_") or hasattr(
                    obj, "coef_"
                )

                candidate_models.append({
                    "variable_name": name,
                    "class": type(obj).__name__,
                    "module": type(obj).__module__,
                    "appears_fitted": is_fitted,
                    "feature_count": getattr(
                        obj, "n_features_in_", None
                    ),
                })
    except Exception:
        continue

models_df = pd.DataFrame(candidate_models)

if models_df.empty:
    print(
        "No scikit-learn estimators found in the current kernel. "
        "Run the classical ML training cells or inspect the notebook "
        "for saved model paths."
    )
else:
    display(models_df)


No scikit-learn estimators found in the current kernel. Run the classical ML training cells or inspect the notebook for saved model paths.


In [20]:

from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

search_roots = [
    project_root / "models",
    project_root / "data" / "processed" / "classical_ml",
    project_root / "reports" / "final_artifacts",
]

extensions = {
    ".joblib", ".pkl", ".pickle", ".sav",
    ".onnx", ".json", ".csv",
}

matches = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if any(part in {".venv", ".git", "__pycache__",
                        ".ipynb_checkpoints"} for part in path.parts):
            continue

        name = path.name.lower()

        if (
            path.suffix.lower() in extensions
            or any(term in name for term in (
                "random_forest", "randomforest",
                "svm", "logistic", "classical_ml",
                "model_comparison",
            ))
        ):
            matches.append({
                "file": str(path.relative_to(project_root)),
                "extension": path.suffix.lower(),
                "size_bytes": path.stat().st_size,
            })

results = pd.DataFrame(matches).drop_duplicates()

if results.empty:
    print("No matching saved model or classical ML artifact files found.")
else:
    display(results.sort_values("file"))

print("\nSearch complete. No files were changed.")


,file,extension,size_bytes
21,data/processed/classical_ml/classical_ml_model...,.csv,488
17,data/processed/classical_ml/classical_ml_per_d...,.csv,1587
16,data/processed/classical_ml/classical_ml_perfo...,.png,90688
15,data/processed/classical_ml/classical_ml_predi...,.csv,94866
19,data/processed/classical_ml/classical_ml_selec...,.csv,438
20,data/processed/classical_ml/final_classical_ml...,.csv,481
18,data/processed/classical_ml/random_forest_feat...,.csv,290
22,data/processed/classical_ml/random_forest_feat...,.csv,1427
3,models/deep_learning/cnn_lstm/cnn_lstm_test_pr...,.csv,210365
2,models/deep_learning/cnn_lstm/cnn_lstm_trainin...,.json,682



Search complete. No files were changed.


In [21]:

from pathlib import Path
import json
import re
import pandas as pd

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

notebook_dir = project_root / "notebooks"

patterns = {
    "Random Forest": r"RandomForestClassifier|RandomForestRegressor",
    "SVM": r"SVC\(|LinearSVC\(|SVR\(|SVC\b",
    "Logistic Regression": r"LogisticRegression",
    "Model fitting": r"\.fit\s*\(",
    "Model saving": r"joblib\.dump|pickle\.dump|save_model",
}

matches = []

if not notebook_dir.exists():
    raise FileNotFoundError(f"Notebook directory not found: {notebook_dir}")

for notebook_path in sorted(notebook_dir.glob("*.ipynb")):
    if ".ipynb_checkpoints" in notebook_path.parts:
        continue

    try:
        notebook = json.loads(
            notebook_path.read_text(encoding="utf-8")
        )
    except Exception as exc:
        print(f"Could not read {notebook_path.name}: {exc}")
        continue

    for cell_index, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        found = [
            label
            for label, pattern in patterns.items()
            if re.search(pattern, source, flags=re.IGNORECASE)
        ]

        if found:
            matches.append({
                "notebook": notebook_path.name,
                "cell_index": cell_index,
                "matches": ", ".join(found),
                "code_preview": source[:450].replace("\n", " "),
            })

results = pd.DataFrame(matches)

if results.empty:
    print("No matching training code found in notebooks.")
else:
    with pd.option_context(
        "display.max_colwidth", 120,
        "display.max_rows", 100
    ):
        display(results)

    output_path = (
        project_root / "reports" / "final_artifacts" /
        "classical_ml_training_code_index.csv"
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    results.to_csv(output_path, index=False)
    print("Index saved:", output_path)


,notebook,cell_index,matches,code_preview
0,07_feature_selection.ipynb,1,Random Forest,from pathlib import Path import numpy as np import pandas as pd import matplotlib.pyplot as plt from sklearn.featu...
1,07_feature_selection.ipynb,17,"Random Forest, Model fitting",# ============================================================ # Random Forest Feature Importance # ================...
2,09_classical_ml.ipynb,1,"Random Forest, SVM, Logistic Regression",from pathlib import Path import numpy as np import pandas as pd import matplotlib.pyplot as plt from sklearn.prepr...
3,09_classical_ml.ipynb,8,"Logistic Regression, Model fitting","logistic_model = LogisticRegression( max_iter=3000, class_weight=""balanced"", random_state=42 ) logistic..."
4,09_classical_ml.ipynb,9,"SVM, Model fitting","svm_model = SVC( kernel=""rbf"", class_weight=""balanced"", probability=True, random_state=42 ) svm_mod..."
5,09_classical_ml.ipynb,10,"Random Forest, Model fitting","rf_model = RandomForestClassifier( n_estimators=300, random_state=42, n_jobs=-1, class_weight=""balan..."


Index saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports/final_artifacts/classical_ml_training_code_index.csv


In [22]:

from pathlib import Path
import json

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

notebook_path = project_root / "notebooks" / "09_classical_ml.ipynb"

if not notebook_path.exists():
    raise FileNotFoundError(f"Notebook not found: {notebook_path}")

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "train_test_split",
    "StandardScaler",
    "LabelEncoder",
    "LogisticRegression",
    "SVC(",
    "RandomForestClassifier",
    ".fit(",
    ".predict(",
    "classical_ml_predictions",
    "final_classical_ml_comparison",
]

for i, cell in enumerate(notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(keyword in source for keyword in keywords):
        print("\n" + "=" * 90)
        print(f"CELL {i}")
        print("=" * 90)
        print(source)



CELL 1
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Imports completed.")

CELL 5
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["diagnosis"])
y_test = label_encoder.transform(test_df["diagnosis"])

class_names = label_encoder.classes_

print("Disease classes:")
print(class_names)

print("\nNumber of classes:", len(class_names))

print("\nEncoded training labels:")
print(np.bincount(y_train))

print("\nEncoded testing labels:")
print(np.bincount(y_test))

CELL 7
scaler = StandardScaler()

X_train_scaled =

In [23]:

from pathlib import Path
import json

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

notebook_path = project_root / "notebooks" / "09_classical_ml.ipynb"

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "StandardScaler",
    "X_train_scaled",
    "selected_feature_names",
    "label_encoder",
    "train_test_split",
    "X_train =",
    "X_test =",
    "X_train_scaled =",
]

for i, cell in enumerate(notebook.get("cells", [])):
    source = "".join(cell.get("source", []))
    if cell.get("cell_type") != "code":
        continue

    if any(keyword in source for keyword in keywords):
        print("\n" + "=" * 80)
        print(f"CELL {i}")
        print("=" * 80)
        print(source)

        outputs = cell.get("outputs", [])
        for output in outputs:
            text = "".join(
                output.get("text", [])
            ) if "text" in output else ""

            if text.strip():
                print("\nSaved output:")
                print(text[:1200])



CELL 1
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Imports completed.")

Saved output:
Imports completed.


CELL 5
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["diagnosis"])
y_test = label_encoder.transform(test_df["diagnosis"])

class_names = label_encoder.classes_

print("Disease classes:")
print(class_names)

print("\nNumber of classes:", len(class_names))

print("\nEncoded training labels:")
print(np.bincount(y_train))

print("\nEncoded testing labels:")
print(np.bincount(y_test))

Saved output:
D

In [2]:

required = [
    "train_df",
    "test_df",
    "selected_features_df",
    "selected_feature_names",
    "scaler",
    "label_encoder",
    "logistic_model",
    "svm_model",
    "rf_model",
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(f"Missing variables: {missing}")

print("Training objects are available.")
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Selected features:", len(selected_feature_names))
print("Classes:", list(label_encoder.classes_))

assert len(train_df) == 4142, "Unexpected training row count"
assert len(test_df) == 2756, "Unexpected test row count"
assert len(selected_feature_names) == 33, "Unexpected feature count"


RuntimeError: Missing variables: ['train_df', 'test_df', 'selected_features_df', 'selected_feature_names', 'scaler', 'label_encoder', 'logistic_model', 'svm_model', 'rf_model']